# 🧬 Human Cytokine Dictionary: an interactive transcriptomics practical

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/garridolab/cytokines-practical/blob/main/CytokineDictionary_Practical_v3_interactive.ipynb)

**Based on:** Oesinghaus *et al.* (2025), *A single-cell cytokine dictionary of human peripheral blood*, bioRxiv
([doi:10.64898/2025.12.12.693897](https://doi.org/10.64898/2025.12.12.693897)).

In the paper, ~9.7 million human blood cells (PBMCs) from **12 donors** were stimulated with **90 cytokines** (or with a PBS control)
and read out by single-cell RNA-seq. Here you work with a **pseudobulk** version of the data: for every *donor × cytokine × cell type*,
the counts of all the cells were **added up** into one bulk-like expression profile. We keep **four cell types** (CD4 T cells, CD8 T cells, NK cells, CD14 monocytes),
**87 cytokines** and the **PBS control**.

### 🎛 You will not write any code
This is an **interactive** practical. Every exercise gives you a few **controls** (menus, sliders, boxes) and a **figure that redraws by itself** each time you change one.
Your job is to **play, observe and interpret**: what changes? why?

| Part | Exercises | Time |
|:--:|---|:--:|
| **A · Python warm-up** | **1** Variables · **2** Basic plots | 12 min |
| **B · Meet the data** | **3** The AnnData object · **4** Distributions with boxplots | 18 min |
| **C · Normalization** | **5** Raw counts vs CPM · **6** The logarithm | 18 min |
| **D · Patterns in the data** | **7** Heatmaps · **8** PCA: the idea · **9** PCA: all samples · **10** PCA: one cell type | 38 min |
| **E · Differential expression** | **11** Volcano plots · **12** Comparing two responses | 22 min |
| | Wrap-up and quiz | 7 min |

### How to use this notebook

| | |
|:--:|---|
| ▶ | **Run a cell** with the ▶ button on its left (or click it and press **Shift + Enter**). Run the **setup cell** below first, and then each exercise cell *once*: the controls appear. |
| 🎛 | **Change a control and wait a second**: the figures and tables update automatically. You never need to run the exercise cell again. |
| 🖱 | In a list, **Ctrl/⌘ + click** selects several items. In text boxes, press **Enter** after typing. |
| ❓ | Each exercise ends with **questions**. Think (or discuss with your neighbour) *before* opening the 💡 answer. |
| 🔁 | If something looks odd, re-run the exercise cell. If the notebook was restarted, run the setup cell again. |

> 💡 In Google Colab the code of the grey cells is hidden on purpose. If you are curious, double-click the cell title to see it.


---
# ⚙️ Setup: run this cell first

It installs what is missing, downloads the data (≈ 500 MB, about a minute on Colab), normalizes it, runs the PCAs and loads the differential-expression
results that were computed beforehand with DESeq2. **You only run it once** (and again if the notebook restarts). Wait until it prints **✅ Ready**.

In [ ]:
#@title ▶ Setup: run this cell first (about 2 minutes) { display-mode: "form" }
# ── Install whatever is missing (Google Colab already has most packages) ──────────────────────────
import importlib.util, shutil, subprocess, sys

REQUIRED = {                       # import name -> PyPI package name
    "numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "matplotlib": "matplotlib", "anndata": "anndata",
    "ipywidgets": "ipywidgets", "jinja2": "jinja2", "gdown": "gdown", "adjustText": "adjustText",
}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"📦 Installing: {', '.join(missing)} …")
    if shutil.which("uv"):
        cmd = ["uv", "pip", "install", "--python", sys.executable, *missing]
    else:
        cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
    subprocess.check_call(cmd)
    importlib.invalidate_caches()


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  00_core.py: imports, constants, colours and plot style
#  (The files of tools/lib are concatenated, in alphabetical order, into the hidden setup cell of the notebook.)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
import os, re, io, sys, json, time, math, html, hashlib, difflib, warnings, itertools
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

warnings.filterwarnings("ignore")


# ── the dataset ────────────────────────────────────────────────────────────────────────────────────
GDRIVE_FILE_ID = "1thFoQnNVVqgKvspg5b5k2fBgjk4bEqAn"          # pseudobulk .h5ad on Google Drive ("anyone with the link")
DATA_FILENAME = "pseudobulks_4ct_proteincoding.h5ad"
DE_CACHE_FILENAME = "de_results.npz"                           # pre-computed DESeq2 results (all cytokines x all cell types)
DE_CACHE_URL = "https://raw.githubusercontent.com/garridolab/cytokines-practical/main/cache/" + DE_CACHE_FILENAME

DISPLAY = {}      # display names of a few words that are also data keys (filled in by the Spanish build)


def disp(word):
    return DISPLAY.get(word, word)


CELL_TYPES = ["CD4_T_cell", "CD8_T_cell", "NK_CD56low", "CD14_Mono"]
CT_LABEL = {"CD4_T_cell": "CD4 T cell", "CD8_T_cell": "CD8 T cell", "NK_CD56low": "NK CD56low", "CD14_Mono": "CD14 monocyte"}
CT_SHORT = {"CD4_T_cell": "CD4 T", "CD8_T_cell": "CD8 T", "NK_CD56low": "NK", "CD14_Mono": "Mono"}
CT_COLORS = {"CD4_T_cell": "#2a78d6", "CD8_T_cell": "#008300", "NK_CD56low": "#eda100", "CD14_Mono": "#e87ba4"}
CT_MARKERS = {"CD4_T_cell": "o", "CD8_T_cell": "s", "NK_CD56low": "^", "CD14_Mono": "D"}
DONORS = [f"Donor{i}" for i in range(1, 13)]

# the expression filter of the practical: at least 10 counts in at least 12 samples (12 = the size of one group)
MIN_COUNT, MIN_SAMPLES = 10, 12
N_HVG = 2000                                                    # highly variable genes used for every PCA
PADJ_DEFAULT, LFC_DEFAULT = 0.05, 0.5                           # default thresholds of the volcano plots

# ── colours: "direction" is blue (down) < grey < red (up) ─────────────────────────────────────────────
UP_COLOR, DOWN_COLOR, NS_COLOR, PBS_COLOR = "#e34948", "#2a78d6", "#c3c2b7", "#8a8984"
BLUE, ORANGE, GREEN, YELLOW, PINK, PURPLE = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"
DARK = "#2b2a28"
DIVERGING = LinearSegmentedColormap.from_list("div", [DOWN_COLOR, "#f0efec", UP_COLOR])
SEQUENTIAL = LinearSegmentedColormap.from_list("seq", ["#f4f8fd", "#86b6ef", "#2a78d6", "#0d366b"])
SEQ_POINTS = LinearSegmentedColormap.from_list("seqp", ["#cde2fb", "#2a78d6", "#0d366b"])
DONOR_HUES = [BLUE, ORANGE, GREEN, YELLOW, PINK, PURPLE]       # 12 donors = 6 hues x 2 marker shapes
CYT_HUES = [BLUE, ORANGE, GREEN, YELLOW, PINK, PURPLE, "#008300", "#8a5a00"]
CYT_MARKERS = ["o", "s", "^", "D", "v", "P", "X", "h"]

plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 11, "axes.labelsize": 10, "legend.frameon": False,
    "legend.fontsize": 8.5, "xtick.labelsize": 9, "ytick.labelsize": 9, "figure.autolayout": False,
})


# ══ 05_immune_genes.py: gene list of the DE exercises ═══

# ── the genes of the differential-expression exercises (11, 12): immune / cytokine-signalling genes ────────────────────────
# Union of four gene sets from Enrichr: GO BP "Cytokine-Mediated Signaling Pathway" (GO:0019221), GO BP "T Cell Activation" (GO:0042110),
# MSigDB Hallmark "Interferon Alpha Response" and "Interferon Gamma Response" (542 symbols; ~506 are expressed in our data).
IMMUNE_GENES = """
ACTN4 ADAR ADIPOQ ADIPOR1 ADIPOR2 AIM2 AKT1 APOL6 APPL1 APPL2 ARID5B ARL4A AUTS2 B2M BANK1 BATF2 BIRC2 BIRC3
BPGM BST2 BTG1 C1R C1S CARD14 CASP1 CASP3 CASP4 CASP7 CASP8 CBL CCL1 CCL11 CCL13 CCL14 CCL15 CCL16 CCL18 CCL19
CCL2 CCL21 CCL22 CCL23 CCL24 CCL25 CCL26 CCL3 CCL3L1 CCL4 CCL4L1 CCL5 CCL7 CCL8 CCM2 CCR1 CCR2 CCR7 CCRL2 CD1C
CD2 CD247 CD27 CD274 CD276 CD28 CD300LF CD38 CD3D CD3E CD4 CD40 CD44 CD47 CD69 CD7 CD70 CD74 CD80 CD84 CD86
CD8A CD8B CDKN1A CEACAM1 CEACAM21 CEBPA CFB CFH CHD7 CHUK CIITA CISH CLCF1 CLEC7A CLSTN1 CMKLR1 CMPK2 CMTR1
CNP CNTF CNTFR COMMD7 CRLF1 CRLF2 CRTAM CSF1 CSF1R CSF2 CSF2RA CSF2RB CSF3 CSF3R CSNK2B CTPS1 CTR9 CTSG CUL4A
CX3CL1 CXADR CXCL10 CXCL11 CXCL12 CXCL9 CXCR2 DCAF12 DDOST DDX58 DDX60 DHX58 DLL4 DPP4 DUOX1 DUOX2 EBI3 EDA
EDA2R EDAR EDN2 EGR1 EIF2AK2 EIF2AK4 EIF4E3 EIF5AL1 ELF1 EPOR EPSTI1 EPX EREG F2RL1 F3 FAS FCER1G FCGR1A FER
FGL2 FKBP1A FLT3 FOXC1 FOXO3 FOXP3 FPR1 GATA3 GBP2 GBP4 GBP6 GCH1 GH1 GHR GMPR GPR18 GPR35 GPR75 GPR89A GRB2
GREM2 GZMA HAX1 HCK HCLS1 HELZ2 HERC6 HIF1A HLA-A HLA-B HLA-C HLA-DMA HLA-DQA1 HLA-DRB1 HLA-G HSPD1 ICAM1
ICOSLG IDO1 IFI27 IFI30 IFI35 IFI44 IFI44L IFIH1 IFIT1 IFIT2 IFIT3 IFITM1 IFITM2 IFITM3 IFNA1 IFNA10 IFNA14
IFNA16 IFNA17 IFNA2 IFNA21 IFNA4 IFNA5 IFNA6 IFNA7 IFNA8 IFNAR2 IFNB1 IFNE IFNGR1 IFNGR2 IFNK IFNLR1 IFNW1
IKBKB IL10 IL10RA IL10RB IL11 IL11RA IL12A IL12B IL12RB1 IL12RB2 IL13 IL13RA1 IL13RA2 IL15 IL15RA IL16 IL17F
IL17RA IL17RC IL18 IL18BP IL18R1 IL18RAP IL1A IL1B IL1F10 IL1R1 IL1RAP IL1RL1 IL2 IL20RA IL20RB IL21R IL22RA1
IL22RA2 IL23R IL2RA IL2RB IL2RG IL3 IL31RA IL33 IL34 IL36A IL36B IL36G IL37 IL3RA IL4 IL4R IL5 IL5RA IL6 IL6R
IL6ST IL7 IL7R IL9 IL9R ILK INS IRAK1 IRAK2 IRAK3 IRAK4 IRF1 IRF2 IRF4 IRF5 IRF7 IRF8 IRF9 IRS1 ISG15 ISG20
ISOC1 ITGB7 ITK JAG2 JAGN1 JAK1 JAK2 JAK3 JAML KIF13B KIT KLRK1 LAMP3 LAP3 LATS2 LCK LCP2 LEP LEPROT LFNG
LGALS3BP LIFR LILRA1 LILRA2 LILRA3 LILRA4 LILRA5 LILRA6 LILRB1 LILRB2 LILRB4 LILRB5 LIMS1 LMBR1L LPAR6 LRP8
LY6E LY9 LYN LYSMD2 MAP3K7 MAPK3 MARCHF1 MDK METTL7B MICA MICB MOV10 MSN MT2A MT3 MTHFD2 MVB12A MVP MX1 MX2
MYD88 NAMPT NCK1 NCK2 NCOA3 NCOA7 NCOR2 NFKB1 NFKBIA NHEJ1 NLRC3 NLRC5 NMI NOD1 NUB1 NUMBL NUP93 OAS1 OAS2
OAS3 OASL OGFR OSMR P2RY14 P3R3URF P4HB PARP12 PARP14 PARP9 PDE4B PDGFB PELI1 PF4 PFKP PIK3CG PIK3R1 PIK3R2
PIK3R3 PIM1 PLA2G4A PLCB1 PLP2 PLSCR1 PLVAP PML PNP PNPT1 PPP3CA PPP3CB PRKCA PRL PRLR PROCR PSMA2 PSMA3
PSMB10 PSMB2 PSMB8 PSMB9 PSME1 PSME2 PTGS2 PTK2B PTPN1 PTPN11 PTPN2 PTPN22 PTPN6 PTPRC PTPRJ PYCARD RAB29
RAPGEF6 RASGRP1 RBCK1 RBX1 RELA RHEX RHOH RIPK1 RIPK2 RNF213 RNF31 RPS6KA4 RPS6KA5 RRAGA RSAD2 RTP4 SAMD9
SAMD9L SAMHD1 SCRIB SECTM1 SELL SELP SERPING1 SIRT1 SLA2 SLAMF6 SLAMF7 SLAMF9 SLC25A28 SMAD4 SOCS1 SOCS2 SOCS3
SOCS4 SOCS5 SOD2 SOS1 SOX4 SP110 SPI1 SPNS1 SPPL2A SRC SRI SSPN ST18 ST3GAL5 ST8SIA4 STAT1 STAT2 STAT3 STAT4
STAT5A STAT5B STAT6 SYK TANK TAP1 TAPBP TARP TDRD7 TENT5A TFF2 TMEM140 TMSB4X TNF TNFAIP2 TNFAIP3 TNFAIP6
TNFRSF11A TNFRSF1A TNFRSF1B TNFRSF4 TNFSF10 TNFSF11 TNFSF14 TNFSF18 TNFSF4 TNIP2 TOLLIP TOR1B TOX4 TP53 TRADD
TRAF1 TRAF3 TRAF3IP2 TRAF6 TRAFD1 TREML2 TRIM14 TRIM21 TRIM25 TRIM26 TRIM5 TSLP TXNDC17 TXNIP TYK2 UBA7 UBE2L6
UGCG UPP1 USP18 VAMP5 VAMP8 VCAM1 WARS1 XAF1 XCL1 XCL2 ZAP70 ZBP1 ZNFX1
""".split()


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  10_data.py: download, tidy-up and pre-computation (everything slow happens here, once)
# ══════════════════════════════════════════════════════════════════════════════════════════════════

# ── files ───────────────────────────────────────────────────────────────────────────────────────────
def _search_dirs():
    here = os.getcwd()
    dirs = [here, os.path.join(here, "cache"), os.path.dirname(here), "/content", os.path.expanduser("~")]
    return [d for d in dict.fromkeys(dirs) if os.path.isdir(d)]


def find_file(filename):
    """Path of `filename` if it already exists next to the notebook (or in /content on Colab), else None."""
    for d in _search_dirs():
        p = os.path.join(d, filename)
        if os.path.exists(p):
            return p
    return None


def download_dir():
    return "/content" if os.path.isdir("/content") else os.getcwd()


def get_h5ad(path=None, verbose=True):
    """Path to the pseudobulk .h5ad; downloads it from Google Drive the first time."""
    path = path or find_file(DATA_FILENAME)
    if path and os.path.exists(path):
        return path
    import gdown
    path = os.path.join(download_dir(), DATA_FILENAME)
    if verbose:
        print("⬇️  Downloading the data from Google Drive (≈ 500 MB, about a minute) …")
    out = gdown.download(f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}", path, quiet=not verbose)
    if out is None or not os.path.exists(path):
        raise RuntimeError("The download failed (Google Drive quota?). Wait a minute and run this cell again, or ask the instructor for the file.")
    return path


def get_de_cache(verbose=True):
    """Path to the pre-computed DESeq2 results, or None if they cannot be found/downloaded (the notebook then computes on demand)."""
    p = find_file(DE_CACHE_FILENAME)
    if p:
        return p
    try:
        import urllib.request
        target = os.path.join(download_dir(), DE_CACHE_FILENAME)
        if verbose:
            print("⬇️  Downloading the pre-computed differential-expression results …")
        urllib.request.urlretrieve(DE_CACHE_URL, target)
        return target
    except Exception as err:                                      # offline, or the file is not published (yet)
        if verbose:
            print(f"ℹ️  Pre-computed DE results not available ({type(err).__name__}): they will be computed when needed (≈ 15 s each).")
        return None


# ── tidy-up of the AnnData object ─────────────────────────────────────────────────────────────────
def is_expressed(counts, min_count=MIN_COUNT, min_samples=MIN_SAMPLES):
    """For a samples × genes matrix of raw counts: True for the genes with at least `min_count` counts in at least `min_samples` samples."""
    return (np.asarray(counts) >= min_count).sum(axis=0) >= min_samples


def load_adata(path):
    """Read the .h5ad and return the tidy AnnData of the practical:
    X = raw counts (dense), obs = sample metadata, var = gene metadata, layers: `cpm` and `log2cpm` (normalised versions of X)."""
    raw = ad.read_h5ad(path)
    # 1) the PBS control was measured in 6 wells and the file stores each well *plus* their total: keep the total only
    is_pbs_well = ((raw.obs["cytokine"] == "PBS") & raw.obs["bc1_well"].notna()).values
    raw = raw[~is_pbs_well]
    # 2) sample metadata, with the order of the categories fixed, and a unique readable name:  donor|cell type|cytokine
    obs = raw.obs[["donor", "cytokine", "treatment", "cell_type", "psbulk_n_cells"]].copy()
    cytokines = ["PBS"] + sorted(c for c in obs["cytokine"].astype(str).unique() if c != "PBS")
    obs["donor"] = pd.Categorical(obs["donor"].astype(str), categories=DONORS)
    obs["cytokine"] = pd.Categorical(obs["cytokine"].astype(str), categories=cytokines)
    obs["cell_type"] = pd.Categorical(obs["cell_type"].astype(str), categories=CELL_TYPES)
    obs["treatment"] = pd.Categorical(np.where(obs["cytokine"] == "PBS", "control", "treated"), categories=["control", "treated"])
    obs["psbulk_n_cells"] = obs["psbulk_n_cells"].astype(int)
    obs.index = (obs["donor"].astype(str) + "|" + obs["cell_type"].astype(str) + "|" + obs["cytokine"].astype(str)).values
    assert obs.index.is_unique
    # 3) the matrix: raw counts, as a plain dense table of numbers
    X = raw.X.toarray() if hasattr(raw.X, "toarray") else np.asarray(raw.X)
    X = np.ascontiguousarray(X, dtype=np.float32)
    var = pd.DataFrame(index=pd.Index(raw.var_names.astype(str), name=None))
    del raw
    # 4) our own columns: library size (obs) and total counts / detection (var)
    obs["library_size"] = X.sum(axis=1, dtype=np.float64).astype(np.int64)
    var["total_counts"] = X.sum(axis=0, dtype=np.float64).astype(np.int64)
    var["n_samples_detected"] = (X > 0).sum(axis=0).astype(int)
    adata = ad.AnnData(X=X, obs=obs, var=var)
    # 5) normalised versions of the matrix, kept in `layers` (more matrices with the same shape as X)
    lib = obs["library_size"].values.astype(np.float64)[:, None]
    cpm = (X / lib * 1e6).astype(np.float32)
    adata.layers["cpm"] = cpm
    adata.layers["log2cpm"] = np.log2(cpm + 1).astype(np.float32)
    return adata


# ── PCA ────────────────────────────────────────────────────────────────────────────────────────────
def pca_svd(M, n_components=10):
    """PCA of a samples × genes matrix: centre → SVD → project. Returns scores, % of variance explained, loadings (genes × PCs)."""
    Mc = M.astype(np.float64) - M.mean(axis=0)                    # 1) centre every gene
    U, S, Vt = np.linalg.svd(Mc, full_matrices=False)             # 2) SVD
    for k in range(min(n_components, len(S))):                    # the sign of a PC is arbitrary: make it reproducible
        j = np.argmax(np.abs(Vt[k]))
        if Vt[k, j] < 0:
            Vt[k], U[:, k] = -Vt[k], -U[:, k]
    scores = (U * S)[:, :n_components]                            # 3) projection: coordinates of the samples on the PCs
    var_pct = (S ** 2 / np.sum(S ** 2) * 100)[:n_components]
    return scores, var_pct, Vt[:n_components].T


def r2_categorical(y, groups):
    """Fraction of the variance of y explained by group membership (one-way ANOVA R²)."""
    y, g = np.asarray(y, float), np.asarray(groups).astype(str)
    ss_tot = ((y - y.mean()) ** 2).sum()
    codes, inv = np.unique(g, return_inverse=True)
    sums, counts = np.bincount(inv, weights=y), np.bincount(inv)
    return float((counts * (sums / counts - y.mean()) ** 2).sum() / ss_tot)


def pc_r2_table(scores, obs, n_pc=10):
    """R² of every PC with every sample variable (categories: ANOVA R²; numeric: squared Pearson correlation)."""
    loglib = np.log10(obs["library_size"].astype(float).values)
    logn = np.log10(obs["psbulk_n_cells"].astype(float).values)
    cols = {
        "cell type": lambda y: r2_categorical(y, obs["cell_type"]),
        "donor": lambda y: r2_categorical(y, obs["donor"]),
        "cytokine": lambda y: r2_categorical(y, obs["cytokine"]),
        "control vs treated": lambda y: r2_categorical(y, obs["treatment"]),
        "library size": lambda y: np.corrcoef(y, loglib)[0, 1] ** 2,
        "number of cells": lambda y: np.corrcoef(y, logn)[0, 1] ** 2,
    }
    n_pc = min(n_pc, scores.shape[1])
    return pd.DataFrame({k: [f(scores[:, i]) if np.std(scores[:, i]) > 0 else 0.0 for i in range(n_pc)] for k, f in cols.items()},
                        index=[f"PC{i + 1}" for i in range(n_pc)])


def make_pca(adata, mask=None, n_hvg=N_HVG, expressed=None):
    """PCA on the `n_hvg` most variable genes (log2 CPM + 1) of the selected samples. Returns a dict with everything the plots need."""
    mask = np.ones(adata.n_obs, bool) if mask is None else np.asarray(mask)
    L = adata.layers["log2cpm"][mask]
    ok = np.ones(adata.n_vars, bool) if expressed is None else np.asarray(expressed)
    cand = np.flatnonzero(ok)
    var_g = L[:, cand].var(axis=0)
    hvg = cand[np.argsort(var_g)[::-1][:n_hvg]]
    scores, var_pct, loadings = pca_svd(L[:, hvg])
    obs = adata.obs[mask]
    return dict(mask=mask, obs=obs, scores=scores, var=var_pct, loadings=loadings, hvg=hvg,
                hvg_names=adata.var_names[hvg].to_numpy(), r2=pc_r2_table(scores, obs))


def ensure_pydeseq2():
    """pydeseq2 is only needed if the pre-computed results are missing: install it the first time it is needed."""
    import importlib.util, subprocess
    if importlib.util.find_spec("pydeseq2") is None:
        print("📦 Installing pydeseq2 …")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pydeseq2>=0.5.4"])
        importlib.invalidate_caches()


# ── differential expression (DESeq2) ──────────────────────────────────────────────────────────────────
def de_gene_universe(adata, expressed=None):
    """The genes that take part in the DE exercises: the immune list ∩ the genes expressed in the whole data set (≈ 500)."""
    expressed = is_expressed(adata.X) if expressed is None else expressed
    return [g for g in adata.var_names[expressed] if g in set(IMMUNE_GENES)]


def run_deseq2(adata, cell_type, treated, reference="PBS", n_cpus=1, genes=None):
    """DESeq2 (pydeseq2): `treated` vs `reference` in one cell type, 12 donors, design `~ donor + cytokine`, restricted to `genes`
    (default: the immune genes, see de_gene_universe). One row per gene of `adata` (NaN for the genes that were not tested).
    This is what produced the cached results."""
    genes = de_gene_universe(adata) if genes is None else genes
    ensure_pydeseq2()
    from pydeseq2.dds import DeseqDataSet
    from pydeseq2.ds import DeseqStats
    names = [f"{d}|{cell_type}|{c}" for c in (reference, treated) for d in DONORS]          # 12 reference + 12 treated samples
    rows = adata.obs_names.get_indexer(names)
    counts = pd.DataFrame(np.rint(adata.X[rows][:, adata.var_names.get_indexer(genes)]).astype(int), index=names, columns=genes)   # RAW counts, integers
    counts = counts.loc[:, is_expressed(counts.values)]                                      # expression filter on these 24 samples
    meta = pd.DataFrame({"donor": DONORS * 2, "cytokine": [reference] * 12 + [treated] * 12}, index=names)
    dds = DeseqDataSet(counts=counts, metadata=meta, design="~ donor + cytokine", quiet=True, n_cpus=n_cpus)
    dds.deseq2()                                                                             # size factors → dispersions → model fit
    stat = DeseqStats(dds, contrast=["cytokine", treated, reference], quiet=True, n_cpus=n_cpus)
    stat.summary()                                                                           # Wald test + Benjamini-Hochberg
    res = stat.results_df.rename(columns={"log2FoldChange": "log2fc", "lfcSE": "lfc_se", "pvalue": "pval", "baseMean": "base_mean"})
    return res[["base_mean", "log2fc", "lfc_se", "pval", "padj"]].reindex(adata.var_names)


CACHE_FIELDS = ("lfc", "nlp_adj")


def de_to_arrays(res):
    """Compact (float16) version of a DESeq2 table, for the cache: log2 fold change and -log10(adjusted p-value)."""
    f16 = lambda v: np.asarray(v, dtype=np.float64).astype(np.float16)
    return dict(lfc=f16(res["log2fc"].clip(-60, 60)), nlp_adj=f16(-np.log10(res["padj"].clip(lower=1e-300))))


class DEResults:
    """All the differential-expression results (cytokine vs PBS, per cell type), pre-computed once with DESeq2.
    `get(cell_type, cytokine)` returns a table with one row per gene; if the cache file is missing, it is computed on demand."""

    def __init__(self, adata, cache_path=None, verbose=True):
        self.adata, self.live, self._memo = adata, {}, {}
        self.arrays = None
        if cache_path and os.path.exists(cache_path):
            z = np.load(cache_path, allow_pickle=False)
            genes = pd.Index(z["genes"].astype(str))
            take = genes.get_indexer(adata.var_names)                              # re-order the genes like `adata`
            self.cell_types = [str(c) for c in z["cell_types"]]
            self.cytokines = [str(c) for c in z["cytokines"]]
            self.arrays = {k: self._reindex(z[k], take) for k in CACHE_FIELDS}
            self.meta = json.loads(str(z["meta"])) if "meta" in z.files else {}
            if verbose:
                print(f"✅ Pre-computed DESeq2 results: {len(self.cell_types)} cell types × {len(self.cytokines)} cytokines")
        else:
            self.cell_types, self.cytokines, self.meta = list(CELL_TYPES), [c for c in adata.obs["cytokine"].cat.categories if c != "PBS"], {}

    @staticmethod
    def _reindex(a, take):
        out = np.full(a.shape[:2] + (len(take),), np.nan, dtype=np.float16)
        ok = take >= 0
        out[..., ok] = a[..., take[ok]]
        return out

    @property
    def cached(self):
        return self.arrays is not None

    def get(self, cell_type, cytokine):
        key = (cell_type, cytokine)
        if key in self._memo:
            return self._memo[key]
        if self.cached and cell_type in self.cell_types and cytokine in self.cytokines:
            i, j = self.cell_types.index(cell_type), self.cytokines.index(cytokine)
            nlp_adj = self.arrays["nlp_adj"][i, j].astype(np.float64)
            res = pd.DataFrame({"log2fc": self.arrays["lfc"][i, j].astype(np.float32), "padj": 10.0 ** -nlp_adj,
                                "neglog10_padj": nlp_adj}, index=self.adata.var_names)
        else:                                                                       # not in the cache: run DESeq2 now (≈ 15 s)
            res = run_deseq2(self.adata, cell_type, cytokine, n_cpus=2)[["log2fc", "padj"]].copy()
            res["neglog10_padj"] = -np.log10(res["padj"].clip(lower=1e-300))
        res["tested"] = res["padj"].notna()
        self._memo[key] = res
        return res

    def lfc_matrix(self, cell_type):
        """cytokines × genes matrix of log2 fold changes in one cell type (cache only)."""
        return self.arrays["lfc"][self.cell_types.index(cell_type)].astype(np.float32) if self.cached else None


def classify_de(res, padj_thresh=PADJ_DEFAULT, lfc_thresh=LFC_DEFAULT):
    """'up' / 'down' / 'ns' for every gene of a DE table (`ns` also for the genes that were not tested)."""
    sig = res["tested"] & (res["padj"] < padj_thresh)
    direction = pd.Series("ns", index=res.index)
    direction[sig & (res["log2fc"] > lfc_thresh)] = "up"
    direction[sig & (res["log2fc"] < -lfc_thresh)] = "down"
    return direction


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  20_widgets.py: the small toolbox every exercise is built from (controls, validation, layout, figures)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
STRICT = False            # tests set this to True so that errors are raised instead of shown as a friendly message
FIG_SINK = None           # tests set this to a function(fig) that receives every figure shown (to save a PNG)
ERRORS = []               # every error caught while redrawing (the students see a friendly message; the tests read this list)

CSS = """
<style>
.cx-panel {border:1px solid rgba(127,127,127,.40); border-radius:8px; padding:8px 12px 6px 12px; background:rgba(127,127,127,.08); margin:2px 0 8px 0;}
.cx-title {font-weight:700; font-size:14px; margin:0 0 4px 0;}
.cx-label {font-weight:600; font-size:12px; margin:0 0 1px 2px;}
.cx-hint  {font-size:11px; opacity:.7; margin:0 0 2px 2px;}
.cx-warn  {border-left:4px solid #eda100; background:rgba(237,161,0,.14); padding:6px 10px; margin:4px 0; border-radius:4px; font-size:13px;}
.cx-err   {border-left:4px solid #e34948; background:rgba(227,73,72,.12); padding:6px 10px; margin:4px 0; border-radius:4px; font-size:13px;}
.cx-info  {border-left:4px solid #2a78d6; background:rgba(42,120,214,.10); padding:6px 10px; margin:4px 0; border-radius:4px; font-size:13px;}
.cx-code  {font-family:monospace; font-size:13px; line-height:1.45; white-space:pre; border:1px solid rgba(127,127,127,.35); border-radius:6px; padding:8px 12px; background:rgba(127,127,127,.07); overflow-x:auto;}
.cx-kw {color:#7a4fd6; font-weight:600} .cx-str {color:#1a8f5a} .cx-num {color:#d6571f} .cx-com {color:#8a8984; font-style:italic}
table.cx-table {border-collapse:collapse; font-size:12.5px; margin:2px 0;}
table.cx-table th {text-align:right; padding:3px 9px; border-bottom:2px solid rgba(127,127,127,.5); font-weight:600;}
table.cx-table td {text-align:right; padding:2px 9px; border-bottom:1px solid rgba(127,127,127,.2);}
table.cx-table td.l, table.cx-table th.l {text-align:left;}
table.cx-table tr:hover td {background:rgba(127,127,127,.12);}
.cx-up {color:#e34948; font-weight:600} .cx-down {color:#2a78d6; font-weight:600}
</style>
"""
STYLE = {"description_width": "initial"}


# ── messages shown in the output area (never a traceback) ───────────────────────────────────────────
def msg(text, kind="warn"):
    """Show a coloured message box. kind: 'warn' (orange), 'err' (red) or 'info' (blue)."""
    display(HTML(f'<div class="cx-{kind}">{text}</div>'))


# ── control builders ───────────────────────────────────────────────────────────────────────────────
def _lay(width=None, **kw):
    return widgets.Layout(width=width, **kw) if width else widgets.Layout(**kw)


def field(label, widget, hint=None):
    """A control with a small bold label above it (and an optional grey hint below)."""
    parts = [widgets.HTML(f'<div class="cx-label">{label}</div>'), widget]
    if hint:
        parts.append(widgets.HTML(f'<div class="cx-hint">{hint}</div>'))
    return widgets.VBox(parts, layout=_lay(margin="0 14px 4px 0"))


def dropdown(options, value=None, width="230px"):
    return widgets.Dropdown(options=options, value=value, layout=_lay(width), style=STYLE)


def toggle(options, value=None, width=None):
    return widgets.ToggleButtons(options=options, value=value, style={"button_width": width or "auto"})


def int_slider(lo, hi, value, step=1, width="260px"):
    return widgets.IntSlider(min=lo, max=hi, value=value, step=step, continuous_update=False, layout=_lay(width), style=STYLE)


def float_slider(lo, hi, value, step=0.1, width="260px", fmt=".2f"):
    return widgets.FloatSlider(min=lo, max=hi, value=value, step=step, continuous_update=False, readout_format=fmt, layout=_lay(width), style=STYLE)


def checkbox(label, value=False, width="260px"):
    return widgets.Checkbox(value=value, description=label, indent=False, layout=_lay(width))


def text_input(value="", width="320px", placeholder=""):
    return widgets.Text(value=value, placeholder=placeholder, continuous_update=False, layout=_lay(width), style=STYLE)


def multi_select(options, value, rows=6, width="200px"):
    return widgets.SelectMultiple(options=options, value=tuple(value), rows=rows, layout=_lay(width), style=STYLE)


def selection_summary(select, label_fn=lambda v: v, max_items=8):
    """A small line under a multi-select that spells out what is selected (the list only shows its first rows)."""
    box = widgets.HTML()

    def _update(*_):
        vals = list(select.value)
        shown = ", ".join(html.escape(str(label_fn(v))) for v in vals[:max_items]) + (f" … (+{len(vals) - max_items})" if len(vals) > max_items else "")
        box.value = f'<div class="cx-hint"><b>{len(vals)}</b> selected: {shown if vals else "none"}</div>'
    select.observe(_update, "value")
    _update()
    return box


def gene_combo(value="ACTB", width="200px"):
    """One gene symbol with auto-completion (type a few letters, pick from the list, press Enter)."""
    return widgets.Combobox(value=value, options=list(GENE_NAMES), ensure_option=False, continuous_update=False,
                            placeholder="gene symbol", layout=_lay(width), style=STYLE)


def geneset_dropdown(text_widget, default_label="— choose a ready-made list —", width="260px"):
    """A dropdown of gene sets that, when used, writes the genes of the set into `text_widget`."""
    dd = dropdown([(default_label, "")] + [(k, k) for k in GENESETS], "", width)

    def _on(change):
        if change["new"]:
            text_widget.value = ", ".join(GENESETS[change["new"]])
            dd.value = ""                                         # back to the placeholder, so the same set can be chosen again
    dd.observe(_on, "value")
    return dd


def panel(title, *rows, note=None):
    """Titled grey box with rows of controls (each row = a widget or a list of widgets)."""
    kids = [widgets.HTML(f'<div class="cx-title">🎛 {title}</div>')]
    for r in rows:
        kids.append(widgets.HBox(list(r), layout=_lay(flex_flow="row wrap", align_items="flex-start")) if isinstance(r, (list, tuple)) else r)
    if note:
        kids.append(widgets.HTML(f'<div class="cx-hint">{note}</div>'))
    box = widgets.VBox(kids)
    box.add_class("cx-panel")
    return box


class Exercise:
    """Controls + a function that redraws the outputs every time a control changes (`widgets.interactive_output`).
    `render(**values)` receives the current value of every control (by name) and draws with matplotlib / display()."""

    def __init__(self, controls, render, ui_rows, title="Controls", note=None, extra=()):
        self.render, self.paused = render, False
        self.controls = dict(controls)
        self._tick = widgets.IntText(value=0)                         # (never displayed) bumping it redraws once, after several changes
        self.panel = panel(title, *ui_rows, note=note)
        self.output = widgets.interactive_output(self._call, {**self.controls, "_tick": self._tick})
        self.ui = widgets.VBox([widgets.HTML(CSS), self.panel, *extra, self.output])

    def values(self):
        return {k: w.value for k, w in self.controls.items()}

    def set_values(self, **values):
        """Change several controls and redraw ONCE."""
        self.paused = True
        try:
            for k, v in values.items():
                self.controls[k].value = v
        finally:
            self.paused = False
        self._tick.value += 1

    def _call(self, _tick=0, **kw):
        if self.paused:
            return
        try:
            self.render(**kw)
        except Exception as err:                                  # a student must never see a traceback
            ERRORS.append(f"{type(err).__name__}: {err}")
            if STRICT:
                raise
            msg(f"⚠️ Something went wrong with this combination ({html.escape(type(err).__name__)}: {html.escape(str(err))[:200]}). Try changing a control, or go back to the default values.", "err")

    def show(self):
        display(self.ui)
        return None


# ── figures ────────────────────────────────────────────────────────────────────────────────────────
def show_fig(fig):
    """Show a figure in the current output and free its memory."""
    if FIG_SINK is not None:
        FIG_SINK(fig)
    plt.show()
    plt.close(fig)


def fmt_n(x):
    return f"{x:,.0f}"


def si(x):
    """12,345,678 → '12.3 M'"""
    x = float(x)
    for lim, s in ((1e9, "G"), (1e6, "M"), (1e3, "k")):
        if abs(x) >= lim:
            return f"{x / lim:.1f} {s}"
    return f"{x:.0f}"


def html_table(df, index=True, float_fmt="{:,.2f}", classes="cx-table", left_cols=(), index_label=None, max_rows=200):
    """Small, clean HTML table (works in Jupyter, Colab, VS Code)."""
    df = df.head(max_rows)
    cols = list(df.columns)
    head = "".join(f'<th class="{"l" if c in left_cols else ""}">{html.escape(str(c))}</th>' for c in cols)
    if index:
        head = f'<th class="l">{html.escape(index_label or "")}</th>' + head
    rows = []
    for idx, r in df.iterrows():
        cells = []
        for c in cols:
            v = r[c]
            if isinstance(v, (float, np.floating)):
                s = "—" if not np.isfinite(v) else (float_fmt.format(v))
            elif isinstance(v, (int, np.integer)):
                s = f"{int(v):,}"
            else:
                s = str(v) if str(v).startswith("<") else html.escape(str(v))          # allow our own <span> markup
            cells.append(f'<td class="{"l" if c in left_cols else ""}">{s}</td>')
        rows.append("<tr>" + (f'<td class="l"><b>{html.escape(str(idx))}</b></td>' if index else "") + "".join(cells) + "</tr>")
    return f'<table class="{classes}"><thead><tr>{head}</tr></thead><tbody>{"".join(rows)}</tbody></table>'


def show_table(df, **kw):
    display(HTML(html_table(df, **kw)))


# ── genes: parsing and validation (names are case-insensitive; unknown ones get a friendly hint) ─────────────
GENE_NAMES, GENE_LOOKUP, GENESETS = [], {}, {}


def parse_genes(text, max_genes=60):
    """'stat1, Ifit3  nonsense' → (['STAT1', 'IFIT3'], ['nonsense'])"""
    valid, unknown = [], []
    for tok in [t for t in re.split(r"[,\s;]+", str(text).strip()) if t]:
        name = GENE_LOOKUP.get(tok.upper())
        if name is None:
            unknown.append(tok)
        elif name not in valid:
            valid.append(name)
    return valid[:max_genes], unknown


def unknown_genes_message(unknown):
    if not unknown:
        return
    parts = []
    for t in unknown[:6]:
        close = difflib.get_close_matches(t.upper(), list(GENE_LOOKUP), n=3, cutoff=0.7)
        parts.append(f"<b>{html.escape(t)}</b>" + (f" (did you mean {', '.join(GENE_LOOKUP[c] for c in close)}?)" if close else ""))
    msg("Gene not found in this dataset: " + "; ".join(parts) + ". Check the spelling (gene symbols, e.g. <i>STAT1</i>).", "warn")


def one_gene(text, default=None):
    """Validate a single gene symbol; shows a hint and returns `default` (possibly None) if it is not valid."""
    valid, unknown = parse_genes(text, 1)
    unknown_genes_message(unknown if not valid else [])
    return valid[0] if valid else default


def gene_index(gene):
    return adata.var_names.get_loc(gene)


def sample_mask(cell_types=None, cytokines=None, donors=None):
    """Boolean mask over the samples of `adata` (None = no restriction)."""
    m = np.ones(adata.n_obs, bool)
    if cell_types is not None:
        m &= adata.obs["cell_type"].isin(list(cell_types)).values
    if cytokines is not None:
        m &= adata.obs["cytokine"].isin(list(cytokines)).values
    if donors is not None:
        m &= adata.obs["donor"].isin(list(donors)).values
    return m


def cyt_label(c):
    """Cytokine names as they are written in the data (IFN-gamma, IL-1-beta) → nicer for titles: IFN-γ, IL-1β."""
    c = re.sub(r"(IL-\d+)-(alpha|beta)", lambda m: m.group(1) + {"alpha": "α", "beta": "β"}[m.group(2)], str(c))        # IL-1-beta → IL-1β
    return c.replace("-gamma", "-γ").replace("-beta", "-β").replace("-alpha", "-α").replace("-omega", "-ω").replace("-lambda", "-λ").replace("-epsilon", "-ε")


def pick_default(options, wanted, fallback_n=3):
    """The wanted values that exist in `options` (so a default never breaks if the data change)."""
    ok = [w for w in wanted if w in options]
    return ok or list(options)[:fallback_n]


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 1: variables and print()
# ══════════════════════════════════════════════════════════════════════════════════════════════════
def highlight_python(code):
    """Tiny syntax highlighter (strings, numbers, comments, keywords) → HTML for the code panel."""
    out = []
    pattern = re.compile(r"(#.*$)|(f?'[^']*'|f?\"[^\"]*\")|(\b\d+(?:\.\d+)?\b)|(\b(?:print|len|True|False)\b)")
    for line in html.escape(code, quote=False).split("\n"):
        def sub(m):
            for cls, g in zip(("com", "str", "num", "kw"), m.groups()):
                if g:
                    return f'<span class="cx-{cls}">{g}</span>'
        out.append(pattern.sub(sub, line))
    return "\n".join(out)


def ex1_program(message, n_donors, cytokine, genes_text, is_control, n_cells, umis_per_cell):
    """The Python program that corresponds to the current values of the controls (as text)."""
    genes = [g.strip() for g in re.split(r"[,;]", genes_text) if g.strip()]
    return "\n".join([
        "# 1) Create variables: a NAME on the left of  =  and a VALUE on the right",
        f"message = {message!r}",
        f"n_donors = {int(n_donors)}",
        f"cytokine = {cytokine!r}",
        f"genes = {genes!r}",
        f"is_control = {bool(is_control)}",
        f"n_cells = {int(n_cells)}",
        f"umis_per_cell = {float(umis_per_cell)!r}",
        "",
        "# 2) Use them: print() shows a value on the screen",
        "print(message)",
        "print('Donors:', n_donors)",
        "print(cytokine, 'was added to', n_donors, 'donors')",
        "print('Number of genes:', len(genes))",
        "",
        "# 3) Variables can be combined to create new variables",
        "library_size = n_cells * umis_per_cell",
        "print('Library size:', library_size)",
        "print('Is this sample a control?', is_control)",
    ])


def ex1_run(program):
    """Run the (safe: built by ex1_program from repr() literals) program; returns what it printed and its variables."""
    buf, ns = io.StringIO(), {}
    real_stdout = sys.stdout
    sys.stdout = buf
    try:
        exec(program, {"__builtins__": {"print": print, "len": len}}, ns)
    finally:
        sys.stdout = real_stdout
    return buf.getvalue(), ns


def ex1_render(message, n_donors, cytokine, genes_text, is_control, n_cells, umis_per_cell):
    program = ex1_program(message, n_donors, cytokine, genes_text, is_control, n_cells, umis_per_cell)
    printed, ns = ex1_run(program)
    badge = {"str": "#1a8f5a", "int": "#d6571f", "float": "#d6571f", "bool": "#7a4fd6", "list": "#2a78d6"}
    rows = "".join(
        f'<tr><td class="l"><b>{k}</b></td><td class="l">{html.escape(repr(v))}</td>'
        f'<td class="l"><span style="color:{badge.get(type(v).__name__, "#888")};font-weight:600">{type(v).__name__}</span></td></tr>'
        for k, v in ns.items())
    table = f'<table class="cx-table"><thead><tr><th class="l">name</th><th class="l">value stored</th><th class="l">type</th></tr></thead><tbody>{rows}</tbody></table>'
    display(HTML(
        '<div style="display:flex; flex-wrap:wrap; gap:16px; align-items:flex-start">'
        f'<div style="flex:1 1 440px"><div class="cx-label">🐍 The Python code written by your controls</div><div class="cx-code">{highlight_python(program)}</div></div>'
        '<div style="flex:1 1 320px">'
        f'<div class="cx-label">🖨️ What <code>print()</code> shows</div><div class="cx-code" style="background:rgba(42,120,214,.10)">{html.escape(printed)}</div>'
        f'<div class="cx-label" style="margin-top:10px">🗃️ What Python remembers (its variables)</div>{table}</div></div>'))


def exercise_variables():
    c = dict(
        message=text_input("Hello, immunologists!", "300px"),
        n_donors=int_slider(1, 12, 12, width="240px"),
        cytokine=dropdown(["IFN-gamma", "IL-4", "IL-6", "TNF-alpha", "IL-2", "IL-10"], "IFN-gamma", "170px"),
        genes_text=text_input("STAT1, CXCL9, ACTB", "250px"),
        is_control=checkbox("this sample is a control", False, "210px"),
        n_cells=int_slider(100, 5000, 2500, 100, "240px"),
        umis_per_cell=float_slider(1000, 8000, 3000, 250, "240px", ".0f"),
    )
    rows = [[field("message  (text)", c["message"], "any text"), field("cytokine  (text)", c["cytokine"]),
             field("genes  (a list of texts)", c["genes_text"], "separate with commas")],
            [field("n_donors  (whole number)", c["n_donors"]), field("n_cells  (whole number)", c["n_cells"]),
             field("umis_per_cell  (decimal number)", c["umis_per_cell"])],
            [field("is_control  (True / False)", c["is_control"])]]
    ex = Exercise(c, ex1_render, rows, title="Create your variables")
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 2: basic plots (simulated data, independent of the dataset)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
PLOT_TYPES = [("📊 Histogram", "hist"), ("📦 Box plot", "box"), ("📶 Bar chart", "bar"), ("🔵 Scatter plot", "scatter"), ("📈 Line plot (time course)", "line")]
COL_CTRL, COL_TRT = "#8a8984", "#eb6834"


def sim_expression(n, mean, effect, sd, r, seed):
    """Fake expression values (log2 CPM) of ONE gene in `n` donors: without (control) and with (treated) a stimulus.
    `effect` = log2 fold change; `r` = how much a donor's two values resemble each other (same person, same baseline)."""
    rng = np.random.default_rng(int(seed))
    z1 = rng.standard_normal(n)
    z2 = r * z1 + np.sqrt(max(1 - r ** 2, 0)) * rng.standard_normal(n)
    return mean + sd * z1, mean + effect + sd * z2


def ex2_render(plot_type, n, mean, effect, sd, bins, r, tau, seed):
    control, treated = sim_expression(n, mean, effect, sd, r, seed)
    fig, ax = plt.subplots(figsize=(7.4, 4.3))
    if plot_type == "hist":
        lo, hi = min(control.min(), treated.min()), max(control.max(), treated.max())
        edges = np.linspace(lo, hi, bins + 1)
        ax.hist(control, bins=edges, color=COL_CTRL, alpha=0.65, label=disp("control"))
        ax.hist(treated, bins=edges, color=COL_TRT, alpha=0.65, label=disp("treated"))
        ax.axvline(control.mean(), color=COL_CTRL, lw=2, ls="--"); ax.axvline(treated.mean(), color=COL_TRT, lw=2, ls="--")
        ax.set_xlabel("expression (log2 CPM)"); ax.set_ylabel("number of donors"); ax.legend()
        ax.set_title(f"Histogram: how many donors have each value? (dashed = mean)")
        code = f"plt.hist(treated, bins={bins})"
    elif plot_type == "box":
        bp = ax.boxplot([control, treated], widths=0.5, patch_artist=True, showfliers=False, medianprops=dict(color="k", lw=2))
        for patch, col in zip(bp["boxes"], (COL_CTRL, COL_TRT)):
            patch.set(facecolor=col, alpha=0.45)
        rng = np.random.default_rng(1)
        for i, v in enumerate((control, treated), 1):
            ax.scatter(i + rng.uniform(-0.15, 0.15, len(v)), v, s=14, color=(COL_CTRL, COL_TRT)[i - 1], alpha=0.8, zorder=3)
        ax.set_xticks([1, 2]); ax.set_xticklabels([disp("control"), disp("treated")]); ax.set_ylabel("expression (log2 CPM)")
        ax.set_title("Box plot: median (line), middle 50 % (box), range (whiskers)")
        code = "plt.boxplot([control, treated])"
    elif plot_type == "bar":
        means, sds = [control.mean(), treated.mean()], [control.std(ddof=1), treated.std(ddof=1)]
        ax.bar([0, 1], means, yerr=sds, capsize=8, color=(COL_CTRL, COL_TRT), alpha=0.8, width=0.55)
        ax.set_xticks([0, 1]); ax.set_xticklabels([disp("control"), disp("treated")]); ax.set_ylabel("expression (log2 CPM): mean ± SD")
        ax.set_title("Bar chart: one number (the mean) per group")
        code = "plt.bar(['control', 'treated'], [control.mean(), treated.mean()])"
    elif plot_type == "scatter":
        ax.scatter(control, treated, s=26, color=BLUE, alpha=0.75, edgecolor="white", linewidth=0.5)
        lo, hi = min(control.min(), treated.min()) - 0.5, max(control.max(), treated.max()) + 0.5
        ax.plot([lo, hi], [lo, hi], color="k", lw=1, ls="--"); ax.text(hi, hi, " y = x", va="top", ha="right", fontsize=8)
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect("equal")
        rr = np.corrcoef(control, treated)[0, 1] if n > 2 else float("nan")
        ax.set_xlabel("control (log2 CPM)"); ax.set_ylabel("treated (log2 CPM)")
        ax.set_title(f"Scatter plot: one point per donor (correlation r = {rr:.2f})")
        code = "plt.scatter(control, treated)"
    else:                                                          # line plot: a time course
        t = np.linspace(0, 24, 13)
        rng = np.random.default_rng(int(seed) + 1000)
        k = min(int(n), 12)                                         # at most 12 donors as thin lines
        for i in range(k):
            ax.plot(t, mean + (control[i] - mean) + effect * (1 - np.exp(-t / tau)) + rng.normal(0, 0.12 * sd, len(t)), color=COL_TRT, alpha=0.25, lw=1)
        ax.plot(t, mean + effect * (1 - np.exp(-t / tau)), color=COL_TRT, lw=3, label="mean response (treated)")
        ax.axhline(mean, color=COL_CTRL, lw=2, ls="--", label="control (no change)")
        ax.set_xlabel("hours after adding the cytokine"); ax.set_ylabel("expression (log2 CPM)"); ax.legend(loc="best")
        ax.set_title(f"Line plot: expression over time ({k} donors, thin lines)")
        code = "plt.plot(hours, expression)"
    fig.tight_layout()
    show_fig(fig)
    display(HTML(f'<div class="cx-hint">Behind the plot, Python runs just one line:</div><div class="cx-code">{html.escape(code)}</div>'))


def exercise_plotting():
    c = dict(plot_type=dropdown(PLOT_TYPES, "hist", "230px"), n=int_slider(5, 200, 60), mean=float_slider(0, 10, 5, 0.5, fmt=".1f"),
             effect=float_slider(-3, 3, 1.0, 0.25, fmt="+.2f"), sd=float_slider(0.1, 3, 1.0, 0.1, fmt=".1f"), bins=int_slider(3, 60, 20),
             r=float_slider(0, 0.99, 0.7, 0.05), tau=float_slider(1, 12, 4, 1, fmt=".0f"), seed=widgets.IntText(value=1, layout=_lay("70px")))
    dice = widgets.Button(description="🎲 new random draw", layout=_lay("170px"))
    dice.on_click(lambda _b: setattr(c["seed"], "value", c["seed"].value + 1))
    f_bins = field("bins  (histogram)", c["bins"], "how many bars")
    f_r = field("similarity of the two values of a donor  (scatter)", c["r"], "0 = unrelated, 1 = identical pattern")
    f_tau = field("response speed: hours to reach 63 %  (line)", c["tau"])
    rows = [[field("type of plot", c["plot_type"]), field("number of donors", c["n"]), field("mean expression (control)", c["mean"])],
            [field("effect of the cytokine (log2 fold change)", c["effect"], "+1 = twice as much, −1 = half"), field("variability between donors (SD)", c["sd"]),
             f_bins, f_r, f_tau],
            [field("random draw number", c["seed"]), widgets.VBox([widgets.HTML('<div class="cx-label">&nbsp;</div>'), dice])]]
    show_for = {"hist": [f_bins], "scatter": [f_r], "line": [f_tau]}

    def _toggle(*_):
        for f in (f_bins, f_r, f_tau):
            f.layout.display = "none"
        for f in show_for.get(c["plot_type"].value, []):
            f.layout.display = None
    c["plot_type"].observe(_toggle, "value")
    _toggle()
    ex = Exercise(c, ex2_render, rows, title="Simulate a gene and choose how to look at it",
                  note="The data are <b>simulated</b> (no real dataset yet): a gene measured in several donors, without and with a cytokine.")
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 3: exploring the AnnData object (X, obs, var) and subsetting it
# ══════════════════════════════════════════════════════════════════════════════════════════════════
class SamplePicker:
    """Pick samples from the 4,224 sample IDs: three dropdowns narrow the list, and the selection survives the filtering."""
    ALL = "(all)"

    def __init__(self, default, rows=9, width="400px", max_options=400):
        obs = adata.obs
        order = np.lexsort((obs["cytokine"].cat.codes.values, obs["cell_type"].cat.codes.values, obs["donor"].cat.codes.values))
        self.ids = list(obs.index[order])                                       # by donor, then cell type, then cytokine
        self.rank = {s: i for i, s in enumerate(self.ids)}
        self.pos = adata.obs_names.get_indexer(self.ids)
        self.max_options = max_options
        self.chosen = [s for s in default if s in self.rank]
        self.f_donor = dropdown([self.ALL] + DONORS, self.ALL, "130px")
        self.f_ct = dropdown([self.ALL] + CELL_TYPES, self.ALL, "150px")
        self.f_cyt = dropdown([self.ALL] + list(obs["cytokine"].cat.categories), self.ALL, "150px")
        self.select = multi_select([], [], rows=rows, width=width)
        self.value_widget = widgets.Text(value="\n".join(self.chosen))             # (never displayed) the selection, as ONE value that only
        self._busy = False                                                          # changes when the student selects, not when the list is filtered
        self.status = widgets.HTML()
        for f in (self.f_donor, self.f_ct, self.f_cyt):
            f.observe(self._refresh, "value")
        self.select.observe(self._on_select, "value")
        self._refresh()
        self.widget = widgets.VBox([
            widgets.HBox([field("Donor", self.f_donor), field("Cell type", self.f_ct), field("Cytokine", self.f_cyt)]),
            self.select, self.status])

    def _refresh(self, *_):
        obs = adata.obs
        m = np.ones(adata.n_obs, bool)
        if self.f_donor.value != self.ALL:
            m &= (obs["donor"] == self.f_donor.value).values
        if self.f_ct.value != self.ALL:
            m &= (obs["cell_type"] == self.f_ct.value).values
        if self.f_cyt.value != self.ALL:
            m &= (obs["cytokine"] == self.f_cyt.value).values
        matches = [s for s, p in zip(self.ids, self.pos) if m[p]]
        shown = matches[:self.max_options]
        opts = sorted(set(shown) | set(self.chosen), key=self.rank.get)
        self._busy = True
        try:
            self.select.value = ()
            self.select.options = opts
            self.select.value = tuple(s for s in self.chosen if s in opts)
        finally:
            self._busy = False
        extra = f" (showing the first {self.max_options}: narrow it down with the filters)" if len(matches) > self.max_options else ""
        self._info = f"{len(matches):,} samples match the filters{extra}. Ctrl/⌘ + click selects several."
        self._update_status()

    def _update_status(self):
        self.status.value = f'<div class="cx-hint">{self._info} <b>{len(self.chosen)}</b> selected.</div>'

    def _on_select(self, change):
        if self._busy:
            return
        self.chosen = list(change["new"])
        self.value_widget.value = "\n".join(self.chosen)
        self._update_status()


def gene_picker(default, width="330px", placeholder="STAT1, CXCL9, ...", with_sets=True, label="Genes (gene symbols, separated by commas)"):
    """Genes as a list of symbols (editable text) + a ready-made-lists dropdown + an auto-complete box to add one gene."""
    text = text_input(default, width, placeholder)
    sets = geneset_dropdown(text) if with_sets else None
    add = widgets.Combobox(value="", options=list(GENE_NAMES), ensure_option=False, continuous_update=False,
                           placeholder="type a gene, press Enter", layout=_lay("190px"), style=STYLE)

    def _add(change):
        new = change["new"]
        if new:
            g, _unknown = parse_genes(new, 1)
            if g and g[0] not in parse_genes(text.value, 500)[0]:
                text.value = (text.value.strip().rstrip(",") + ", " + g[0]).lstrip(", ")
            add.value = ""
    add.observe(_add, "value")
    second = [field("…or a ready-made list", sets)] if with_sets else []
    return text, widgets.VBox([field(label, text), widgets.HBox(second + [field("…or add one gene" if with_sets else "Add one gene (auto-complete)", add)])])


MATRICES = [("X: raw counts", "X"), ("layer 'cpm': counts per million", "cpm"), ("layer 'log2cpm': log2(CPM + 1)", "log2cpm")]


def anndata_repr(a):
    """The classic text of an AnnData object (we write it ourselves so that it looks the same with every anndata version)."""
    lines = [f"AnnData object with n_obs × n_vars = {a.n_obs} × {a.n_vars}",
             "    obs: " + ", ".join(f"'{c}'" for c in a.obs.columns),
             "    var: " + ", ".join(f"'{c}'" for c in a.var.columns)]
    layers = [k for k in a.layers.keys() if k is not None]
    if layers:
        lines.append("    layers: " + ", ".join(f"'{k}'" for k in layers))
    return "\n".join(lines)


def _anndata_cartoon(sub):
    """Left: the AnnData scheme with the shapes of the selection. Right: where the selection sits in the full matrix."""
    n, m = sub.n_obs, sub.n_vars
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.5, 3.3), gridspec_kw={"width_ratios": [1.55, 1]})
    a1.set_xlim(0, 10); a1.set_ylim(0, 5.6); a1.axis("off")
    box = lambda x, y, w, h, col, txt, tc="white": (a1.add_patch(plt.Rectangle((x, y), w, h, fc=col, ec="white", lw=2)),
                                                     a1.text(x + w / 2, y + h / 2, txt, ha="center", va="center", color=tc, fontsize=10, fontweight="bold"))
    box(2.5, 0.4, 5.2, 3.5, BLUE, f"X\n{n} samples × {m} genes\n(the numbers)")
    box(0.1, 0.4, 2.2, 3.5, ORANGE, f"obs\n{n} samples ×\n{sub.obs.shape[1]} columns\n(about the\nSAMPLES)")
    box(2.5, 4.1, 5.2, 1.3, GREEN, f"var\n{m} genes × {sub.var.shape[1]} columns\n(about the GENES)")
    a1.text(8.0, 2.3, f"layers\n{', '.join(k for k in sub.layers.keys() if k is not None)}\n(same shape as X)", va="center", fontsize=8.5, color=PBS_COLOR)
    a1.set_title("The AnnData object you selected", loc="left")
    # right: where the subset is in the complete matrix
    a2.set_xlim(0, adata.n_vars); a2.set_ylim(adata.n_obs, 0)
    a2.add_patch(plt.Rectangle((0, 0), adata.n_vars, adata.n_obs, fc="#eef3fb", ec=BLUE, lw=1.2))
    ri, ci = adata.obs_names.get_indexer(sub.obs_names), adata.var_names.get_indexer(sub.var_names)
    for r in ri:
        a2.axhline(r, color=ORANGE, lw=1.4, alpha=0.9)
    for c in ci:
        a2.axvline(c, color=GREEN, lw=1.4, alpha=0.9)
    a2.scatter(np.tile(ci, len(ri)), np.repeat(ri, len(ci)), s=14, color=BLUE, zorder=3)
    a2.set_xticks([]); a2.set_yticks([])
    a2.set_xlabel(f"{adata.n_vars:,} genes", fontsize=9); a2.set_ylabel(f"{adata.n_obs:,} samples", fontsize=9)
    a2.spines[:].set_visible(False)
    a2.set_title("…inside the complete matrix", loc="left")
    fig.tight_layout()
    show_fig(fig)


def ex3_render(samples, genes_text, matrix):
    genes, unknown = parse_genes(genes_text, 200)
    unknown_genes_message(unknown)
    samples = [s for s in str(samples).split("\n") if s in adata.obs_names]
    if not samples or not genes:
        msg("Select <b>at least one sample</b> and <b>one gene</b> to create a subset.", "info")
        return
    sub = adata[samples, genes]                                                    # ← the subset: samples first, genes second
    display(HTML(f'<div class="cx-label">The code that creates your subset</div>'
                 f'<div class="cx-code">sub = adata[samples, genes]      <span class="cx-com"># {len(samples)} sample names, {len(genes)} gene names</span>\n'
                 f'print(sub)</div>'
                 f'<div class="cx-code" style="margin-top:4px;background:rgba(42,120,214,.10)">{html.escape(anndata_repr(sub))}</div>'))
    _anndata_cartoon(sub)
    M = sub.X if matrix == "X" else sub.layers[matrix]
    dfx = pd.DataFrame(np.asarray(M), index=sub.obs_names, columns=sub.var_names)
    fmt = "{:,.0f}" if matrix == "X" else ("{:,.1f}" if matrix == "cpm" else "{:.2f}")
    shade = np.log10(dfx.values.astype(float) + 1) if matrix != "log2cpm" else dfx.values.astype(float)
    top = max(float(np.nanmax(shade)), 1e-9) * 1.25
    head = "".join(f'<th>{html.escape(str(c))}</th>' for c in dfx.columns)                      # plain HTML: no jinja2 needed
    body = "".join(
        f'<tr><td class="l"><b>{html.escape(str(idx))}</b></td>' + "".join(
            f'<td style="background:rgba(42,120,214,{0.05 + 0.6 * max(0.0, float(sh)) / top:.2f})">{fmt.format(v)}</td>' for v, sh in zip(row, shrow)) + "</tr>"
        for idx, row, shrow in zip(dfx.index, dfx.values, shade))
    table_x = f'<table class="cx-table"><thead><tr><th class="l"></th>{head}</tr></thead><tbody>{body}</tbody></table>'
    obs_df = sub.obs.copy()
    for c in ("donor", "cytokine", "treatment", "cell_type"):
        obs_df[c] = obs_df[c].astype(str)
    obs_df["psbulk_n_cells"] = obs_df["psbulk_n_cells"].map("{:,}".format); obs_df["library_size"] = obs_df["library_size"].map("{:,}".format)
    var_df = sub.var.copy()
    var_df["total_counts"] = var_df["total_counts"].map("{:,}".format)
    tabs = widgets.Output(), widgets.Output(), widgets.Output()
    with tabs[0]:
        display(HTML(f'<div class="cx-hint">Rows = the {len(samples)} samples (same rows as <code>obs</code>), columns = the {len(genes)} genes (same rows as <code>var</code>).</div>'))
        display(HTML(table_x))
    with tabs[1]:
        display(HTML('<div class="cx-hint">One row per sample: who it comes from, what was added, which cell type, and how many cells / counts it holds.</div>'))
        display(HTML(html_table(obs_df, left_cols=obs_df.columns, index_label="sample ID")))
    with tabs[2]:
        display(HTML('<div class="cx-hint">One row per gene: its total counts in the complete dataset, and in how many of the 4,224 samples it has at least one count.</div>'))
        display(HTML(html_table(var_df, left_cols=["total_counts"], index_label=disp("gene"))))
    tab = widgets.Tab(children=list(tabs))
    for i, t in enumerate(["X  (the numbers)", "obs  (samples)", "var  (genes)"]):
        tab.set_title(i, t)
    display(tab)


def exercise_anndata():
    default = [f"Donor1|{ct}|PBS" for ct in CELL_TYPES] + ["Donor1|CD14_Mono|IFN-gamma"]
    sp = SamplePicker(default)
    gtext, gwidget = gene_picker("LYZ, CD3E, NKG7, STAT1, ACTB")
    matrix = dropdown(MATRICES, "X", "300px")
    c = dict(samples=sp.value_widget, genes_text=gtext, matrix=matrix)
    rows = [widgets.HBox([widgets.VBox([widgets.HTML('<div class="cx-label">① Choose samples (rows)</div>'), sp.widget]),
                          widgets.VBox([widgets.HTML('<div class="cx-label">② Choose genes (columns)</div>'), gwidget,
                                        field("③ Which matrix to look at", matrix, "X holds the raw counts; the layers hold normalised versions (Exercises 5-6)")],
                                       layout=_lay(margin="0 0 0 20px"))],
                         layout=_lay(flex_flow="row wrap"))]
    ex = Exercise(c, ex3_render, rows, title="Build your own subset of the dataset")
    ex.picker = sp
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 4: distributions of raw counts (boxplots of pseudobulk samples)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
SPLITS = [("cell type", "cell_type"), ("control (PBS) vs treated", "treatment"), (disp("donor"), "donor"), ("cytokine (the ones selected below)", "cytokine")]
SPLIT_NAME = {"cell_type": "cell type", "treatment": disp("condition"), "donor": disp("donor"), "cytokine": disp("cytokine")}


def group_colors(split_by, groups):
    if split_by == "cell_type":
        return {g: CT_COLORS[g] for g in groups}
    if split_by == "treatment":
        return {"control": "#8a8984", "treated": ORANGE}
    pal = DONOR_HUES if split_by == "donor" else CYT_HUES
    return {g: pal[i % len(pal)] for i, g in enumerate(groups)}


def group_label(split_by, g):
    return CT_LABEL.get(g, g) if split_by == "cell_type" else (cyt_label(g) if split_by == "cytokine" else disp(g))


def draw_boxes(ax, data, genes, groups, colors, log_y=True, points=True, label_fn=lambda g: g):
    """Boxes of `data[gene][group]` (arrays), grouped by gene on the x axis and coloured by group."""
    G = len(groups)
    w = 0.8 / G
    rng = np.random.default_rng(0)
    floor = 0.5
    for i, gene in enumerate(genes):
        for j, g in enumerate(groups):
            v = np.asarray(data[gene][g], float)
            if len(v) == 0:
                continue
            v = np.maximum(v, floor) if log_y else v
            pos = i + (j - (G - 1) / 2) * w
            bp = ax.boxplot([v], positions=[pos], widths=w * 0.85, patch_artist=True, showfliers=False, medianprops=dict(color="k", lw=1.6),
                            whiskerprops=dict(color="#666", lw=0.9), capprops=dict(color="#666", lw=0.9))
            bp["boxes"][0].set(facecolor=colors[g], alpha=0.55, edgecolor="#555", lw=0.9)
            if points:
                vv = v if len(v) <= 120 else rng.choice(v, 120, replace=False)
                ax.scatter(pos + rng.uniform(-w * 0.3, w * 0.3, len(vv)), vv, s=5, color=colors[g], alpha=0.55, linewidths=0, zorder=3)
    ax.set_xticks(range(len(genes))); ax.set_xticklabels(genes, fontsize=9.5, fontstyle="italic")
    ax.set_xlim(-0.6, len(genes) - 0.4)
    if log_y:
        ax.set_yscale("log")


def ex4_render(genes_text, cell_types, split_by, all_cytokines, cytokines, log_y, show_points):
    genes, unknown = parse_genes(genes_text, 12)
    unknown_genes_message(unknown)
    cell_types, cytokines = list(cell_types), list(cytokines)
    if not genes:
        msg("Type at least one valid <b>gene symbol</b> (for example <i>ACTB, LYZ, CD3E</i>).", "info"); return
    if not cell_types:
        msg("Select at least one <b>cell type</b>.", "info"); return
    if split_by == "cytokine" and not cytokines:
        msg("To split by cytokine, select at least one cytokine in the list (hold Ctrl/⌘ to select several).", "info"); return
    m = sample_mask(cell_types, None if (all_cytokines and split_by != "cytokine") else (cytokines or None))
    if m.sum() == 0:
        msg("No samples match this selection.", "warn"); return
    obs = adata.obs[m]
    if split_by == "cytokine":
        groups = [c for c in obs["cytokine"].cat.categories if c in cytokines][:10]
    else:
        groups = [g for g in obs[split_by].cat.categories if g in set(obs[split_by])]
    keep = obs[split_by].isin(groups).values
    rows = np.flatnonzero(m)[keep]
    labels = obs[split_by].astype(str).values[keep]
    colors = group_colors(split_by, groups)
    cols = [gene_index(g) for g in genes]
    block = adata.X[np.ix_(rows, cols)]
    data = {gn: {g: block[labels == g, k] for g in groups} for k, gn in enumerate(genes)}
    lib = {g: adata.obs["library_size"].values[rows][labels == g].astype(float) for g in groups}
    G, nG = len(groups), len(genes)
    fig, axes = plt.subplots(1, 2, figsize=(max(8.5, 1.15 * nG * max(G, 2) ** 0.5 + 4.2), 4.6), gridspec_kw={"width_ratios": [max(nG * (0.5 + 0.3 * G), 3), max(G, 2)]})
    draw_boxes(axes[0], data, genes, groups, colors, log_y, show_points)
    axes[0].set_ylabel("raw counts of the gene in a sample" + ("  (log scale)" if log_y else ""))
    axes[0].set_title("Each box = the pseudobulk samples of one group")
    ax = axes[1]
    for j, g in enumerate(groups):
        v = lib[g]
        bp = ax.boxplot([v], positions=[j], widths=0.6, patch_artist=True, showfliers=False, medianprops=dict(color="k", lw=1.6))
        bp["boxes"][0].set(facecolor=colors[g], alpha=0.55, edgecolor="#555")
        if show_points:
            rng = np.random.default_rng(j)
            vv = v if len(v) <= 120 else rng.choice(v, 120, replace=False)
            ax.scatter(j + rng.uniform(-0.2, 0.2, len(vv)), vv, s=5, color=colors[g], alpha=0.55, linewidths=0, zorder=3)
    ax.set_xticks(range(G)); ax.set_xticklabels([group_label(split_by, g) for g in groups], rotation=35 if G > 2 else 0, ha="right" if G > 2 else "center", fontsize=8.5)
    ax.set_yscale("log" if log_y else "linear"); ax.set_ylabel("library size (total counts of the sample)"); ax.set_title("Total counts per sample")
    handles = [Line2D([], [], marker="s", ls="", color=colors[g], alpha=0.7, label=f"{group_label(split_by, g)}  (n = {int((labels == g).sum())})") for g in groups]
    fig.legend(handles=handles, loc="lower center", ncol=min(G, 5), bbox_to_anchor=(0.5, -0.06), fontsize=8.5, title=f"colour = {SPLIT_NAME[split_by]}", title_fontsize=9)
    fig.tight_layout()
    show_fig(fig)
    # the numbers behind the boxes
    tab = pd.DataFrame({group_label(split_by, g): [np.median(data[gn][g]) for gn in genes] + [np.median(lib[g])] for g in groups}, index=genes + ["library size"])
    tab = tab.map(lambda x: f"{x:,.0f}") if hasattr(tab, "map") else tab.applymap(lambda x: f"{x:,.0f}")
    display(HTML('<div class="cx-label">Median value of each box (raw counts)</div>' + html_table(tab, index_label=disp("gene"))))


def exercise_boxplots():
    gtext, gwidget = gene_picker("ACTB, GAPDH, LYZ, CD3E, NKG7, STAT1", placeholder="ACTB, LYZ, ...")
    cyts = list(adata.obs["cytokine"].cat.categories)
    c = dict(genes_text=gtext, cell_types=multi_select([(CT_LABEL[x], x) for x in CELL_TYPES], CELL_TYPES, rows=4, width="170px"),
             split_by=dropdown(SPLITS, "cell_type", "270px"), all_cytokines=checkbox("use all 88 conditions", True, "190px"),
             cytokines=multi_select(cyts, pick_default(cyts, ["PBS", "IFN-gamma", "IL-4", "IL-1-beta"]), rows=6, width="170px"),
             log_y=checkbox("log scale (y axis)", True, "170px"), show_points=checkbox("show the samples as dots", True, "200px"))

    def _sync(*_):
        c["cytokines"].disabled = c["all_cytokines"].value and c["split_by"].value != "cytokine"
    c["all_cytokines"].observe(_sync, "value"); c["split_by"].observe(_sync, "value"); _sync()
    rows = [[widgets.VBox([gwidget]), field("Cell types", c["cell_types"]), field("Group the boxes by", c["split_by"]),
             widgets.VBox([field("Cytokines", c["cytokines"]), selection_summary(c["cytokines"], cyt_label), c["all_cytokines"]])], [c["log_y"], c["show_points"]]]
    ex = Exercise(c, ex4_render, rows, title="Which genes, which samples?")
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 5: raw counts vs counts per million (CPM)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
VIEWS5 = [("Real data: same donors, PBS vs a cytokine", "pair"), ("Real data: all samples of a cell type", "all"), ("Toy example (simulated)", "toy")]


def _geo_ratio(a, b):
    """Typical (geometric-mean) ratio b / a over donors, with +1 so that zeros do not break it."""
    return float(np.exp(np.mean(np.log((np.asarray(b, float) + 1) / (np.asarray(a, float) + 1)))))


def _ratio_text(r, what):
    if r >= 1.15:
        return f'<b>{r:.1f}× higher</b>'
    if r <= 1 / 1.15:
        return f'<b>{1 / r:.1f}× lower</b>'
    return '<b>about the same</b>'


def _paired(ax, a, b, labels, title, colors=(PBS_COLOR, ORANGE), log=True):
    for x, y in zip(a, b):
        ax.plot([0, 1], [x, y], color=UP_COLOR if y > x else DOWN_COLOR, lw=1.1, alpha=0.65, zorder=1)
    ax.scatter(np.zeros(len(a)), a, s=26, color=colors[0], zorder=2)
    ax.scatter(np.ones(len(b)), b, s=26, color=colors[1], zorder=2)
    ax.plot([0, 1], [np.exp(np.mean(np.log(np.maximum(a, 0.5)))), np.exp(np.mean(np.log(np.maximum(b, 0.5))))], color="k", lw=3, marker="D", ms=7, zorder=3)
    ax.set_xticks([0, 1]); ax.set_xticklabels(labels); ax.set_xlim(-0.4, 1.4)
    if log:
        ax.set_yscale("log")
    ax.set_title(title)


def ex5_pair(gene, cell_type, cytokine):
    names = lambda c: [f"{d}|{cell_type}|{c}" for d in DONORS]
    r0, r1 = adata.obs_names.get_indexer(names("PBS")), adata.obs_names.get_indexer(names(cytokine))
    j = gene_index(gene)
    lib0, lib1 = adata.obs["library_size"].values[r0].astype(float), adata.obs["library_size"].values[r1].astype(float)
    raw0, raw1 = adata.X[r0, j].astype(float), adata.X[r1, j].astype(float)
    cpm0, cpm1 = adata.layers["cpm"][r0, j].astype(float), adata.layers["cpm"][r1, j].astype(float)
    cl = cyt_label(cytokine)
    display(HTML(
        f'<div class="cx-info">In <b>{CT_LABEL[cell_type]}</b>, with {cl} the library size is {_ratio_text(_geo_ratio(lib0, lib1), "")} than with PBS. '
        f'The raw counts of <i>{gene}</i> are {_ratio_text(_geo_ratio(raw0, raw1), "")}. '
        f'After CPM, <i>{gene}</i> is {_ratio_text(_geo_ratio(cpm0, cpm1), "")} ({cl} / PBS, typical donor).</div>'))
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.9))
    lab = ["PBS", cl]
    _paired(axes[0], lib0, lib1, lab, "① Library size of the sample"); axes[0].set_ylabel("total counts of the sample")
    _paired(axes[1], raw0, raw1, lab, f"② RAW counts of {gene}"); axes[1].set_ylabel("raw counts")
    _paired(axes[2], cpm0, cpm1, lab, f"③ CPM of {gene}  (normalised)"); axes[2].set_ylabel("counts per million (CPM)")
    fig.suptitle(f"{CT_LABEL[cell_type]}: one line per donor (red = higher with {cl}, blue = lower).  Black = typical donor", fontsize=10, y=1.0)
    fig.tight_layout()
    show_fig(fig)


def ex5_all(gene, cell_type):
    m = adata.obs["cell_type"].values == cell_type
    j = gene_index(gene)
    lib = adata.obs["library_size"].values[m].astype(float)
    ctrl = (adata.obs["treatment"].values[m] == "control")
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.1), sharex=True)
    for ax, y, name in zip(axes, (adata.X[m, j].astype(float), adata.layers["cpm"][m, j].astype(float)), ("raw counts", "CPM")):
        ax.scatter(lib[~ctrl], y[~ctrl] + 1, s=8, alpha=0.45, color=CT_COLORS[cell_type], linewidths=0, label="cytokine-treated samples")
        ax.scatter(lib[ctrl], y[ctrl] + 1, s=36, facecolors="none", edgecolors="k", linewidths=1.1, label="PBS controls")
        ax.set_xscale("log"); ax.set_yscale("log")
        ok = y >= 0
        r = np.corrcoef(np.log10(lib), np.log10(y + 1))[0, 1] if np.std(y) > 0 else float("nan")
        if np.isfinite(r):
            slope, icpt = np.polyfit(np.log10(lib), np.log10(y + 1), 1)
            xs = np.array([lib.min(), lib.max()]); ax.plot(xs, 10 ** (icpt + slope * np.log10(xs)), color=UP_COLOR, lw=1.8)
        ax.set_xlabel("library size of the sample (total counts)"); ax.set_ylabel(f"{name} + 1")
        ax.set_title(f"{gene}: {name}\ncorrelation with library size: r = {r:+.2f}")
    axes[0].legend(loc="upper left", markerscale=1.6)
    fig.suptitle(f"{CT_LABEL[cell_type]}: {m.sum():,} samples (every dot is a pseudobulk sample)", fontsize=10, y=1.0)
    fig.tight_layout()
    show_fig(fig)


def ex5_toy(depth, fold):
    names = ["gene A", "gene B\n(interferon gene)", "gene C", "gene D"]
    base = np.array([1000.0, 50.0, 2500.0, 1450.0])                    # sample 1: PBS, deep (5,000 counts in total)
    p2 = base / base.sum(); p2[1] *= fold; p2 /= p2.sum()               # sample 2: gene B truly goes `fold` times up, the others do not change
    n2 = base.sum() * depth
    raw = np.vstack([base, np.round(p2 * n2)])
    cpm = raw / raw.sum(axis=1, keepdims=True) * 1e6
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.9))
    x, w = np.arange(4), 0.38
    for ax, M, title, yl in zip(axes, (raw, cpm), ("RAW counts", "COUNTS PER MILLION (CPM)"), ("counts", "CPM")):
        ax.bar(x - w / 2, M[0], w, color=PBS_COLOR, label=f"sample 1: PBS ({raw[0].sum():,.0f} counts in total)")
        ax.bar(x + w / 2, M[1], w, color=ORANGE, label=f"sample 2: with cytokine ({raw[1].sum():,.0f} counts in total)")
        for xi, (a, b) in enumerate(zip(M[0], M[1])):
            ax.text(xi, max(a, b) * 1.03, f"×{b / a:.2f}", ha="center", fontsize=9, fontweight="bold", color=UP_COLOR if b / a > 1.15 else (DOWN_COLOR if b / a < 0.87 else "#333"))
        ax.set_xticks(x); ax.set_xticklabels(names, fontsize=9); ax.set_ylabel(yl); ax.set_title(title)
        ax.set_ylim(0, M.max() * 1.18)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.04), fontsize=9)
    fig.suptitle(f"Only gene B really changes (×{fold:g}). Numbers above the bars: sample 2 / sample 1", fontsize=10, y=1.0)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    show_fig(fig)
    display(HTML(f'<div class="cx-info">Sample 2 was sequenced <b>{depth:.0%}</b> as deeply as sample 1. In raw counts <b>every gene looks '
                 f'{"lower" if depth < 0.95 else "similar"}</b>, even gene B (which truly goes up ×{fold:g}). '
                 f'After dividing by the library size (CPM), genes A, C and D are about unchanged and gene B shows its true increase.</div>'))


def ex5_render(view, gene, cell_type, cytokine, depth, fold):
    if view == "toy":
        return ex5_toy(depth, fold)
    g = one_gene(gene, None)
    if g is None:
        msg("Type a valid <b>gene symbol</b> (for example <i>ACTB</i>, <i>STAT1</i>, <i>IDO1</i>).", "info"); return
    if view == "pair":
        if cytokine == "PBS":
            msg("Choose a cytokine different from PBS.", "info"); return
        ex5_pair(g, cell_type, cytokine)
    else:
        ex5_all(g, cell_type)


def exercise_normalization():
    cyts = [c for c in adata.obs["cytokine"].cat.categories if c != "PBS"]
    c = dict(view=widgets.ToggleButtons(options=VIEWS5, value="pair", style={"button_width": "300px"}, layout=_lay("930px")),
             gene=gene_combo("ACTB", "170px"), cell_type=dropdown([(CT_LABEL[x], x) for x in CELL_TYPES], "CD14_Mono", "190px"),
             cytokine=dropdown([(cyt_label(x), x) for x in cyts], "IFN-gamma", "190px"),
             depth=float_slider(0.1, 1.0, 0.2, 0.05, "280px", ".0%"), fold=float_slider(1, 8, 4, 0.5, "280px", ".1f"))
    f_gene, f_ct, f_cyt = field("Gene", c["gene"], "type a symbol, e.g. ACTB, STAT1, IDO1"), field("Cell type", c["cell_type"]), field("Compare PBS with…", c["cytokine"])
    f_depth = field("Sequencing depth of sample 2 (relative to sample 1)", c["depth"])
    f_fold = field("True fold change of gene B", c["fold"])

    def _toggle(*_):
        real = c["view"].value != "toy"
        for f in (f_gene, f_ct):
            f.layout.display = None if real else "none"
        f_cyt.layout.display = None if c["view"].value == "pair" else "none"
        for f in (f_depth, f_fold):
            f.layout.display = "none" if real else None
    c["view"].observe(_toggle, "value"); _toggle()
    ex = Exercise(c, ex5_render, [[c["view"]], [f_gene, f_ct, f_cyt, f_depth, f_fold]], title="Raw counts or CPM?")
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 6: logarithms: scale, spread and the zeros
# ══════════════════════════════════════════════════════════════════════════════════════════════════
BASES = [("2  (log₂, the one we use)", "2"), ("e  (natural log)", "e"), ("10  (log₁₀)", "10")]
BASE_VALUE = {"2": 2.0, "e": math.e, "10": 10.0}
PSEUDO_OPTIONS = [0, 0.1, 0.5, 1, 2, 5, 10]
_MV_CACHE = {}


def log_transform(cpm, base, pseudo):
    """log_base(cpm + pseudocount). With pseudocount 0 the zeros become NaN (log of 0 does not exist)."""
    with np.errstate(divide="ignore", invalid="ignore"):
        out = np.log(cpm.astype(np.float64) + pseudo) / math.log(BASE_VALUE[base])
    out[~np.isfinite(out)] = np.nan
    return out


def mean_var(M):
    """Mean and variance of every column of M ignoring NaN (columns with fewer than 20 valid values → NaN)."""
    valid = np.isfinite(M).sum(axis=0)
    mean, var = np.nanmean(M, axis=0), np.nanvar(M, axis=0, ddof=1)
    bad = valid < 20
    mean[bad] = np.nan; var[bad] = np.nan
    return mean, var


def top20_share(var):
    v = var[np.isfinite(var)]
    return np.sort(v)[-20:].sum() / v.sum()


def ex6_render(gene, cell_type, base, pseudo):
    g = one_gene(gene, None)
    if g is None:
        msg("Type a valid <b>gene symbol</b> (for example <i>CXCL10</i>, <i>IDO1</i>, <i>ACTB</i>).", "info"); return
    m = adata.obs["cell_type"].values == cell_type
    cpm_all = adata.layers["cpm"][m][:, EXPRESSED]
    key = ("cpm", cell_type)
    if key not in _MV_CACHE:                                                  # the CPM statistics never change: compute them once
        _MV_CACHE[key] = mean_var(cpm_all.astype(np.float64))
    mean_c, var_c = _MV_CACHE[key]
    L = log_transform(cpm_all, base, pseudo)
    mean_l, var_l = mean_var(L)
    j = gene_index(g)
    x = adata.layers["cpm"][m, j].astype(float)
    lx = log_transform(x, base, pseudo)
    n_zero = int((x == 0).sum())
    bname = {"2": "log₂", "e": "ln", "10": "log₁₀"}[base]
    fig, axes = plt.subplots(2, 2, figsize=(10.6, 7.6))
    col = CT_COLORS[cell_type]
    # (a) the gene, in CPM
    ax = axes[0, 0]
    ax.hist(x, bins=40, color=col, alpha=0.85)
    ax.axvline(np.mean(x), color="k", ls="--", lw=1.5); ax.axvline(np.median(x), color=UP_COLOR, ls=":", lw=2)
    ax.set_xlabel("CPM"); ax.set_ylabel("number of samples")
    sk = stats.skew(x)
    ax.set_title(f"① {g} in CPM: skewed (skewness {sk:.1f})")
    ax.legend(handles=[Line2D([], [], color="k", ls="--", label=f"mean {np.mean(x):,.0f}"), Line2D([], [], color=UP_COLOR, ls=":", lw=2, label=f"median {np.median(x):,.0f}")], loc="upper right")
    # (b) the same gene after the logarithm
    ax = axes[0, 1]
    ok = np.isfinite(lx)
    ax.hist(lx[ok], bins=40, color=col, alpha=0.85)
    ax.set_xlabel(f"{bname}(CPM + {pseudo:g})" if pseudo else f"{bname}(CPM)"); ax.set_ylabel("number of samples")
    sk2 = stats.skew(lx[ok]) if ok.sum() > 2 else float("nan")
    ax.set_title(f"② {g} after the logarithm (skewness {sk2:.1f})")
    ax.text(0.02, 0.95, f"variance of the values:\n{np.var(x, ddof=1):,.0f}  →  {np.var(lx[ok], ddof=1):.2f}", transform=ax.transAxes, ha="left", va="top", fontsize=9.5)
    # (c) all genes: variance vs mean, CPM
    ax = axes[1, 0]
    ok = (mean_c > 0.01) & (var_c > 0)
    ax.scatter(mean_c[ok], var_c[ok], s=3, alpha=0.25, color=BLUE, linewidths=0, rasterized=True)
    xx = np.array([0.01, mean_c[ok].max()]); ax.plot(xx, xx ** 2 * 0.3, color="k", ls="--", lw=1.1, label="variance ∝ mean²")
    ax.set_xscale("log"); ax.set_yscale("log"); ax.legend(loc="upper left")
    ax.set_xlabel("mean CPM of the gene"); ax.set_ylabel("variance across samples")
    ax.set_title(f"③ CPM: variance rises with the mean\n20 genes hold {top20_share(var_c):.0%} of the total variance")
    # (d) all genes: variance vs mean, log
    ax = axes[1, 1]
    ok = np.isfinite(mean_l) & np.isfinite(var_l) & (var_l > 0)
    ax.scatter(mean_l[ok], var_l[ok], s=3, alpha=0.25, color=BLUE, linewidths=0, rasterized=True)
    ax.set_xlabel(f"mean of {bname}(CPM{' + ' + format(pseudo, 'g') if pseudo else ''})"); ax.set_ylabel("variance across samples")
    ax.set_title(f"④ Log: variance not tied to expression\n20 genes hold {top20_share(var_l):.1%} of the total variance")
    fig.suptitle(f"{CT_LABEL[cell_type]} · {m.sum():,} samples · {cpm_all.shape[1]:,} expressed genes", fontsize=10.5, y=1.0)
    fig.tight_layout()
    show_fig(fig)
    notes = []
    if pseudo == 0:
        notes.append(f"With <b>no pseudocount</b>, log(0) does not exist: " + (f"the {n_zero} zero values of {g} ({n_zero / len(x):.0%}) cannot be plotted, and " if n_zero else f"{g} has no zeros here, but ") + "genes with many zeros are dropped from panel ④.")
    else:
        notes.append(f"The pseudocount ({pseudo:g}) is added <i>before</i> the logarithm so that zeros become log({pseudo:g}) instead of −∞. Larger pseudocounts squash the low values even more.")
    if base != "2":
        notes.append(f"Changing the base only <b>rescales</b> the axis (by a constant: log₂(x) = {math.log2(BASE_VALUE[base]):.2f} × {bname}(x)): the shape of the histogram does not change, the variances are multiplied by {(1 / math.log2(BASE_VALUE[base])) ** 2:.2f}).")
    display(HTML('<div class="cx-info">' + "<br>".join(notes) + "</div>"))


def exercise_logtransform():
    c = dict(gene=gene_combo("CXCL10", "170px"), cell_type=dropdown([(CT_LABEL[x], x) for x in CELL_TYPES], "CD14_Mono", "190px"),
             base=dropdown(BASES, "2", "220px"),
             pseudo=widgets.SelectionSlider(options=PSEUDO_OPTIONS, value=1, continuous_update=False, layout=_lay("330px"), style=STYLE))
    rows = [[field("Gene (top panels)", c["gene"], "e.g. CXCL10, IDO1, ACTB"), field("Cell type", c["cell_type"]), field("Base of the logarithm", c["base"]),
             field("Pseudocount added before the log", c["pseudo"], "0 = no pseudocount: log(0) is undefined")]]
    ex = Exercise(c, ex6_render, rows, title="Transform the CPM values")
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 7: heatmaps of raw counts vs normalised values
# ══════════════════════════════════════════════════════════════════════════════════════════════════
HEAT_VIEWS = [("Raw counts", "raw"), ("Normalised: log₂(CPM + 1)", "norm"), ("Both, side by side", "both")]
HEAT_COLS = [("every donor (12 columns per condition)", "donors"), ("the mean of the 12 donors", "mean")]
HEAT_DEFAULT_GROUPS = ["Monocyte markers", "T-cell markers", "CD8 T-cell markers", "NK-cell markers", "IFN-γ / STAT1 response",
                       "IL-4 / STAT6 response", "IL-1β / NF-κB response", "IL-2 / STAT5 response"]
HEAT_DEFAULT_CYTOKINES = ["PBS", "IFN-gamma", "IL-4", "IL-1-beta", "IL-2"]
MAX_HEAT_CONDITIONS, MAX_HEAT_GENES = 25, 70


def zscore_rows(M):
    """z-score every ROW (gene) across its columns (samples): (x − mean) / standard deviation."""
    sd = M.std(axis=1, ddof=1, keepdims=True)
    z = (M - M.mean(axis=1, keepdims=True)) / np.where(sd > 0, sd, 1)
    return np.where(sd > 0, z, 0.0)


def heatmap_data(genes, cell_types, conditions, kind, cols_mode):
    """genes × columns matrix (+ library size per column). Columns are ordered: cell type > condition > donor."""
    cols = [f"{d}|{ct}|{cy}" for ct in cell_types for cy in conditions for d in DONORS]
    idx = adata.obs_names.get_indexer(cols)
    gi = [gene_index(g) for g in genes]
    src = adata.X if kind == "raw" else adata.layers["log2cpm"]
    M = np.asarray(src[np.ix_(idx, gi)], dtype=np.float64).T
    lib = np.log10(adata.obs["library_size"].values[idx].astype(float))
    if cols_mode == "mean":
        M = M.reshape(len(genes), -1, len(DONORS)).mean(axis=2)
        lib = lib.reshape(-1, len(DONORS)).mean(axis=1)
    return M, lib[None, :]


def draw_heatmaps(panels, genes, group_sizes, group_names, cell_types, conditions, cols_mode, lib, title):
    """`panels` = [(title, matrix), …]: same genes and columns, one panel each, with a cell-type bar and a library-size bar on top."""
    n_panel = len(panels)
    n_per_ct = len(conditions) * (len(DONORS) if cols_mode == "donors" else 1)
    n_cols = n_per_ct * len(cell_types)
    height = 0.22 * len(genes) + 2.3
    fig = plt.figure(figsize=(6.4 * n_panel + 1.6, height))
    gs = fig.add_gridspec(3, n_panel + 1, height_ratios=[0.7, 0.7, max(len(genes), 4)], width_ratios=[1] * n_panel + [0.025],
                          hspace=0.06, wspace=0.05, top=1 - 0.55 / height, bottom=0.95 / height + 0.04, left=0.12 if n_panel == 1 else 0.075, right=0.94)
    im = None
    for c, (name, M) in enumerate(panels):
        ax_ct, ax_lib, ax = (fig.add_subplot(gs[r, c]) for r in range(3))
        for k, ct in enumerate(cell_types):
            ax_ct.add_patch(plt.Rectangle((k * n_per_ct - 0.5, 0), n_per_ct, 1, color=CT_COLORS[ct], lw=0))
            ax_ct.text(k * n_per_ct + n_per_ct / 2 - 0.5, 0.5, CT_SHORT[ct], ha="center", va="center", fontsize=9, fontweight="bold", color="black" if ct == "NK_CD56low" else "white")
        ax_ct.set_xlim(-0.5, n_cols - 0.5); ax_ct.set_ylim(0, 1); ax_ct.axis("off"); ax_ct.set_title(name, fontsize=11, pad=6)
        ax_lib.imshow(lib, aspect="auto", cmap="Greys", interpolation="nearest")
        ax_lib.set_xticks([]); ax_lib.set_yticks([0] if c == 0 else [])
        if c == 0:
            ax_lib.set_yticklabels(["library size"], fontsize=7.5)
        for sp in ax_lib.spines.values():
            sp.set_visible(False)
        im = ax.imshow(zscore_rows(M), aspect="auto", cmap=DIVERGING, vmin=-2.5, vmax=2.5, interpolation="nearest")
        ax.set_yticks(range(len(genes))); ax.set_yticklabels(genes if c == 0 else [], fontsize=8, fontstyle="italic")
        if cols_mode == "donors":
            ax.set_xticks([(k + 0.5) * len(DONORS) - 0.5 for k in range(len(cell_types) * len(conditions))])
            ax.set_xticklabels([cyt_label(cy) for _ in cell_types for cy in conditions], rotation=90, fontsize=7)
        else:
            ax.set_xticks(range(n_cols)); ax.set_xticklabels([cyt_label(cy) for _ in cell_types for cy in conditions], rotation=90, fontsize=7.5)
        for b in np.cumsum(group_sizes)[:-1] - 0.5:
            ax.axhline(b, color="white", lw=2)
        for k in range(1, len(cell_types)):
            ax.axvline(k * n_per_ct - 0.5, color="white", lw=2)
        for sp in ax.spines.values():
            sp.set_visible(False)
        if c == 0 and group_names:
            starts = np.concatenate([[0], np.cumsum(group_sizes)[:-1]])
            for gname, st, sz in zip(group_names, starts, group_sizes):
                ax.annotate(gname, xy=(0, st + sz / 2 - 0.5), xycoords=("axes fraction", "data"), xytext=(-52, 0), textcoords="offset points",
                            ha="right", va="center", fontsize=7.5, color="#555", annotation_clip=False)
    fig.colorbar(im, cax=fig.add_subplot(gs[2, n_panel]), label="z-score of the gene")
    fig.suptitle(title, y=1 - 0.06 / height, fontsize=10.5, fontweight="bold")
    show_fig(fig)


def ex7_render(view, groups, extra_text, cell_types, cytokines, cols_mode):
    conditions = [c for c in adata.obs["cytokine"].cat.categories if c in set(cytokines)]
    cell_types = [c for c in CELL_TYPES if c in set(cell_types)]
    genes, group_sizes, group_names = [], [], []
    for gname in groups:
        gg = [g for g in GENESETS.get(gname, []) if g not in genes]
        if gg:
            genes += gg; group_sizes.append(len(gg)); group_names.append(gname)
    extra, unknown = parse_genes(extra_text, 40)
    unknown_genes_message(unknown)
    extra = [g for g in extra if g not in genes]
    if extra:
        genes += extra; group_sizes.append(len(extra)); group_names.append("your genes")
    if not genes:
        msg("Select at least one <b>gene group</b> or type a few gene symbols.", "info"); return
    if not conditions or not cell_types:
        msg("Select at least one <b>cell type</b> and one <b>cytokine / condition</b>.", "info"); return
    if len(conditions) > MAX_HEAT_CONDITIONS:
        msg(f"Too many conditions for a readable heatmap: showing the first {MAX_HEAT_CONDITIONS}.", "warn"); conditions = conditions[:MAX_HEAT_CONDITIONS]
    if len(genes) > MAX_HEAT_GENES:
        msg(f"Too many genes for a readable heatmap: showing the first {MAX_HEAT_GENES}.", "warn")
        genes = genes[:MAX_HEAT_GENES]; group_sizes = list(np.minimum(group_sizes, np.maximum(0, MAX_HEAT_GENES - np.concatenate([[0], np.cumsum(group_sizes)[:-1]]))).astype(int))
        group_names = [n for n, s in zip(group_names, group_sizes) if s > 0]; group_sizes = [s for s in group_sizes if s > 0]
    panels, lib = [], None
    for kind, name in (("raw", "BEFORE: raw counts"), ("norm", "AFTER: log₂(CPM + 1)")):
        if view == "both" or view == kind:
            M, lib = heatmap_data(genes, cell_types, conditions, kind, cols_mode)
            panels.append((name, M))
    n_cols = len(cell_types) * len(conditions) * (len(DONORS) if cols_mode == "donors" else 1)
    draw_heatmaps(panels, genes, group_sizes, group_names, cell_types, conditions, cols_mode, lib,
                  f"{len(genes)} genes × {n_cols} columns ({len(cell_types)} cell types × {len(conditions)} conditions" + (" × 12 donors)" if cols_mode == "donors" else ", mean of 12 donors)"))


def exercise_heatmaps():
    cyts = list(adata.obs["cytokine"].cat.categories)
    c = dict(view=widgets.ToggleButtons(options=HEAT_VIEWS, value="both", style={"button_width": "250px"}),
             groups=multi_select(list(GENESETS), pick_default(list(GENESETS), HEAT_DEFAULT_GROUPS), rows=10, width="240px"),
             extra_text=None, cell_types=multi_select([(CT_LABEL[x], x) for x in CELL_TYPES], CELL_TYPES, rows=4, width="170px"),
             cytokines=multi_select(cyts, pick_default(cyts, HEAT_DEFAULT_CYTOKINES), rows=10, width="170px"),
             cols_mode=dropdown(HEAT_COLS, "donors", "290px"))
    ttext, twidget = gene_picker("", "260px", "e.g. TNF, IL1B", with_sets=False, label="Extra genes (optional)")
    c["extra_text"] = ttext
    rows = [[c["view"]],
            [widgets.VBox([field("Gene groups", c["groups"], "Ctrl/⌘ + click: several"), selection_summary(c["groups"], max_items=3)]), widgets.VBox([twidget]),
             widgets.VBox([field("Cell types", c["cell_types"]), field("Columns show", c["cols_mode"])]),
             widgets.VBox([field("Cytokines / conditions", c["cytokines"], "PBS = control"), selection_summary(c["cytokines"], cyt_label)])]]
    ex = Exercise(c, ex7_render, rows, title="What do you want to see?")
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 8: the geometry of PCA with two genes (rotation and projection)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
PAIRS8 = {"corr": ("ISG15", "MX1"), "uncorr": ("IDO1", "IL1B")}
PAIR_CHOICES = [("Correlated genes: ISG15 & MX1 (two interferon-response genes)", "corr"), ("Uncorrelated genes: IDO1 & IL1B (two different pathways)", "uncorr")]


def pair_data(pair):
    """Centred log2(CPM + 1) of the two genes in the monocyte samples + everything PCA would compute from them."""
    a, b = PAIRS8[pair]
    m = adata.obs["cell_type"].values == "CD14_Mono"
    P = np.c_[adata.layers["log2cpm"][m, gene_index(a)], adata.layers["log2cpm"][m, gene_index(b)]].astype(np.float64)
    Pc = P - P.mean(axis=0)
    C = np.cov(Pc.T)
    w, V = np.linalg.eigh(C)                                                  # PCA of two variables: eigen-decomposition of the covariance
    w, V = w[::-1], V[:, ::-1]
    if V[np.argmax(np.abs(V[:, 0])), 0] < 0:
        V[:, 0] = -V[:, 0]
    theta1 = np.degrees(np.arctan2(V[1, 0], V[0, 0])) % 180
    is_ifn = adata.obs.loc[adata.obs["cell_type"] == "CD14_Mono", "cytokine"].astype(str).str.startswith("IFN-").values
    return dict(genes=(a, b), Pc=Pc, C=C, eigvals=w, V=V, theta1=theta1, r=np.corrcoef(Pc.T)[0, 1], is_ifn=is_ifn)


def ex8_render(pair, angle, show_proj, show_pc):
    d = pair_data(pair)
    a, b = d["genes"]
    Pc, C = d["Pc"], d["C"]
    th = np.radians(angle)
    u = np.array([np.cos(th), np.sin(th)])                                   # direction of our axis
    v = np.array([-np.sin(th), np.cos(th)])                                  # the perpendicular one
    s_u, s_v = Pc @ u, Pc @ v                                                # coordinates in the rotated system
    total = np.trace(C)
    captured = u @ C @ u
    best = d["eigvals"][0]
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), gridspec_kw={"width_ratios": [1.15, 1, 1.15]})
    # ① the data with the axis and the projections
    ax = axes[0]
    ax.scatter(Pc[:, 0], Pc[:, 1], s=7, alpha=0.4, color=NS_COLOR, linewidths=0, label="other cytokines and PBS")
    ax.scatter(Pc[d["is_ifn"], 0], Pc[d["is_ifn"], 1], s=9, alpha=0.7, color=UP_COLOR, linewidths=0, label="interferons")
    L = 1.15 * np.abs(Pc).max()
    if show_proj:
        idx = np.random.default_rng(3).choice(len(Pc), 45, replace=False)
        for i in idx:
            p = (Pc[i] @ u) * u
            ax.plot([Pc[i, 0], p[0]], [Pc[i, 1], p[1]], color="k", lw=0.7, ls=":", alpha=0.8)
        ax.scatter(*(np.outer(Pc[idx] @ u, u)).T, s=12, color="k", zorder=5, label="projection on the axis")
    ax.plot([-L * u[0], L * u[0]], [-L * u[1], L * u[1]], color=PURPLE, lw=2.6, zorder=4, label=f"your axis ({angle}°)")
    if show_pc:
        for k, (col, nm) in enumerate(((DARK, "PC1"), ("#888", "PC2"))):
            e = d["V"][:, k] * 1.9 * np.sqrt(d["eigvals"][k])
            ax.annotate("", xy=e, xytext=-e, arrowprops=dict(arrowstyle="<|-|>", color=col, lw=1.6, ls="--"))
            ax.text(*(e * 1.08), nm, fontsize=10, fontweight="bold", color=col, ha="center", va="center")
    ax.axhline(0, color="#ccc", lw=0.6); ax.axvline(0, color="#ccc", lw=0.6)
    ax.set_xlim(-L, L); ax.set_ylim(-L, L); ax.set_aspect("equal")
    ax.set_xlabel(f"{a}  (log₂ CPM + 1, centred)"); ax.set_ylabel(f"{b}  (log₂ CPM + 1, centred)")
    ax.set_title(f"① {a} vs {b} in monocytes (r = {d['r']:.2f})")
    ax.legend(loc="lower right", fontsize=7.5, markerscale=1.4)
    # ② variance of the projections for every possible angle
    ax = axes[1]
    ang = np.arange(0, 181)
    var_ang = np.array([np.array([np.cos(np.radians(t)), np.sin(np.radians(t))]) @ C @ np.array([np.cos(np.radians(t)), np.sin(np.radians(t))]) for t in ang])
    ax.plot(ang, var_ang / total * 100, color=BLUE, lw=2.5)
    ax.scatter([angle], [captured / total * 100], s=90, color=PURPLE, zorder=5)
    ax.annotate(f"{captured / total:.0%}", xy=(angle, captured / total * 100), xytext=(0, 11), textcoords="offset points", ha="center", fontsize=10.5, fontweight="bold", color=PURPLE)
    if show_pc:
        ax.axvline(d["theta1"], color=DARK, ls="--", lw=1.4); ax.text(d["theta1"], 3, f" PC1: {d['theta1']:.0f}° ({best / total:.0%})", rotation=90, va="bottom", ha="right", fontsize=9)
    ax.set_ylim(0, 105); ax.set_xlim(0, 180); ax.set_xticks([0, 45, 90, 135, 180])
    ax.set_xlabel("angle of the axis (degrees)"); ax.set_ylabel("% of the total variance kept by the projection")
    ax.set_title("② How much variance does each axis keep?")
    # ③ the data seen from the rotated axes
    ax = axes[2]
    ax.scatter(s_u, s_v, s=7, alpha=0.4, color=NS_COLOR, linewidths=0)
    ax.scatter(s_u[d["is_ifn"]], s_v[d["is_ifn"]], s=9, alpha=0.7, color=UP_COLOR, linewidths=0)
    ax.axhline(0, color=PURPLE, lw=1.2); ax.axvline(0, color="#ccc", lw=0.6)
    ax.set_xlim(-L, L); ax.set_ylim(-L, L); ax.set_aspect("equal")
    ax.set_xlabel(f"coordinate on your axis  (variance {np.var(s_u, ddof=1):.2f})"); ax.set_ylabel(f"coordinate on the perpendicular axis  (variance {np.var(s_v, ddof=1):.2f})")
    ax.set_title(f"③ The same samples after rotating by {angle}°\ncorrelation between the new coordinates: {np.corrcoef(s_u, s_v)[0, 1]:+.2f}")
    fig.tight_layout()
    show_fig(fig)
    if abs(((angle - d["theta1"] + 90) % 180) - 90) <= 2:
        msg(f"🎯 Your axis is (almost) <b>PC1</b>: it keeps the maximum variance ({best / total:.0%}) and, in ③, the two new coordinates are no longer correlated: "
            f"<b>PCA is just this rotation</b>. Dropping the perpendicular axis (PC2) would lose only {1 - best / total:.0%} of the variance.", "info")


def exercise_pca_geometry():
    c = dict(pair=dropdown(PAIR_CHOICES, "corr", "520px"), angle=widgets.IntSlider(min=0, max=180, value=0, step=1, continuous_update=False,
                                                                                     layout=_lay("430px"), style=STYLE, readout_format="d"),
             show_proj=checkbox("show the projections (dotted lines)", True, "270px"), show_pc=checkbox("show PC1 and PC2", False, "200px"))
    snap = widgets.Button(description="🎯 jump to PC1", layout=_lay("140px"))
    snap.on_click(lambda _b: setattr(c["angle"], "value", int(round(pair_data(c["pair"].value)["theta1"])) % 181))
    rows = [[field("Choose the pair of genes", c["pair"])],
            [field("Rotate your axis: angle (degrees)", c["angle"], "0° = the x axis (first gene), 90° = the y axis (second gene)"),
             widgets.VBox([widgets.HTML('<div class="cx-label">&nbsp;</div>'), snap]), widgets.VBox([widgets.HTML('<div class="cx-label">&nbsp;</div>'), c["show_proj"]]),
             widgets.VBox([widgets.HTML('<div class="cx-label">&nbsp;</div>'), c["show_pc"]])]]
    ex = Exercise(c, ex8_render, rows, title="Rotate an axis and watch the projection")
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercises 9 and 10: PCA scatter plots (all samples, and one cell type at a time)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
def pca_scatter(ax, P, color_by, pcs=(0, 1), cytokines=(), gene=None, s=14):
    """Scatter of two PCs. color_by: cell_type | donor | treatment | cytokine | n_cells | library_size | gene."""
    sc, (a, b), obs = P["scores"], pcs, P["obs"]
    kw = dict(linewidths=0.3, edgecolors="white")
    legend = True
    if color_by == "cell_type":
        for ct in CELL_TYPES:
            m = (obs["cell_type"] == ct).values
            if m.any():
                ax.scatter(sc[m, a], sc[m, b], s=s, alpha=0.8, color=CT_COLORS[ct], marker=CT_MARKERS[ct], label=CT_LABEL[ct], **kw)
    elif color_by == "donor":
        for i, dn in enumerate(DONORS):
            m = (obs["donor"] == dn).values
            ax.scatter(sc[m, a], sc[m, b], s=s, alpha=0.85, color=DONOR_HUES[i % 6], marker="o" if i < 6 else "^", label=dn, **kw)
    elif color_by == "treatment":
        m = (obs["treatment"] == "treated").values
        ax.scatter(sc[m, a], sc[m, b], s=s * 0.8, color=NS_COLOR, alpha=0.6, linewidths=0, label=f"treated with a cytokine ({m.sum():,})", zorder=1)
        ax.scatter(sc[~m, a], sc[~m, b], s=s * 2.6, color="black", label=f"control, PBS ({(~m).sum():,})", zorder=2, **kw)
    elif color_by == "cytokine":
        order = {c: i for i, c in enumerate(adata.obs["cytokine"].cat.categories)}
        picked = sorted(cytokines, key=lambda c: order.get(c, 999))[:len(CYT_HUES)]          # same colour for the same cytokine, whatever the click order
        m = ~obs["cytokine"].isin(["PBS"] + picked).values
        ax.scatter(sc[m, a], sc[m, b], s=s * 0.6, color=NS_COLOR, alpha=0.5, linewidths=0, label="all other cytokines", zorder=1)
        for i, cy in enumerate([c for c in picked if c != "PBS"]):
            m = (obs["cytokine"] == cy).values
            ax.scatter(sc[m, a], sc[m, b], s=s * 1.5, color=CYT_HUES[i], marker=CYT_MARKERS[i], label=cyt_label(cy), zorder=2, **kw)
        m = (obs["cytokine"] == "PBS").values
        ax.scatter(sc[m, a], sc[m, b], s=s * 2.6, color="black", label="PBS (control)", zorder=3, **kw)
    else:                                                                       # a number: n_cells, library_size or a gene
        legend = False
        if color_by == "gene":
            values, label = P_gene_values(P, gene), f"{gene}\nlog₂(CPM + 1)"
        elif color_by == "library_size":
            values, label = np.log10(obs["library_size"].values.astype(float)), "log₁₀ library size"
        else:
            values, label = np.log10(obs["psbulk_n_cells"].values.astype(float)), "log₁₀ number of cells"
        order = np.argsort(values)                                              # highest values on top
        pts = ax.scatter(sc[order, a], sc[order, b], c=values[order], cmap=SEQ_POINTS, s=s * 1.5, **kw)
        plt.colorbar(pts, ax=ax, shrink=0.8, pad=0.01, label=label)
    if legend:
        n_leg = len(ax.get_legend_handles_labels()[0])
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), fontsize=8.5, markerscale=1.5, handletextpad=0.3, columnspacing=1.2, ncol=min(n_leg, 6))
    ax.set_xlabel(f"PC{a + 1}  ({P['var'][a]:.1f} % of the variance)"); ax.set_ylabel(f"PC{b + 1}  ({P['var'][b]:.1f} %)")
    ax.axhline(0, color="#ddd", lw=0.6, zorder=0); ax.axvline(0, color="#ddd", lw=0.6, zorder=0)


def P_gene_values(P, gene):
    return adata.layers["log2cpm"][P["mask"], gene_index(gene)].astype(float)


def plot_scree(ax, P, pcs, n=10):
    var = P["var"][:n]
    ax.bar(range(1, len(var) + 1), var, color=[ORANGE if i in pcs else "#9bbbe6" for i in range(len(var))])
    for i, v in enumerate(var[:6]):
        ax.text(i + 1, v + 0.8, f"{v:.0f}%" if v >= 10 else f"{v:.1f}%", ha="center", fontsize=7.5)
    ax.set_xticks(range(1, len(var) + 1)); ax.set_xlabel("principal component"); ax.set_ylabel("variance explained (%)"); ax.set_title("How much variance does each PC hold?")


def plot_r2(ax, P, pcs, factors=None):
    """What does each of the chosen PCs capture? R² with every sample variable."""
    R2 = P["r2"]
    cols = [c for c in (factors or R2.columns) if c in R2.columns]
    y = np.arange(len(cols))
    w = 0.38
    for k, (pc, col) in enumerate(zip(pcs, (ORANGE, BLUE))):
        vals = [R2.loc[f"PC{pc + 1}", c] for c in cols]
        ax.barh(y + (k - 0.5) * w, vals, w, color=col, label=f"PC{pc + 1}")
        for yy, v in zip(y + (k - 0.5) * w, vals):
            if v > 0.03:
                ax.text(v + 0.01, yy, f"{v:.2f}", va="center", fontsize=7.5)
    ax.set_yticks(y); ax.set_yticklabels([disp(c) for c in cols], fontsize=8.5); ax.invert_yaxis(); ax.set_xlim(0, 1.12)
    ax.set_xlabel("R²  (fraction of the PC explained by the variable)"); ax.set_title("What does each PC capture?"); ax.legend(loc="lower right")


COLOR_CHOICES = [(disp("Donor"), "donor"), ("Cell type", "cell_type"), ("Control vs treated", "treatment"), ("Cytokine (highlight a few)", "cytokine"),
                 ("Number of cells in the sample", "n_cells"), ("Library size", "library_size"), ("Expression of a gene", "gene")]
PC_CHOICES = [(f"PC{i}", i - 1) for i in range(1, 11)]


def _pca_common_msgs(color_by, gene, cytokines):
    if color_by == "gene":
        g = one_gene(gene, None)
        if g is None:
            msg("Type a valid <b>gene symbol</b> to colour by its expression (e.g. <i>LYZ</i>, <i>CD3E</i>, <i>UTY</i>).", "info")
        return g
    if color_by == "cytokine" and len(cytokines) > len(CYT_HUES):
        msg(f"Only the first {len(CYT_HUES)} cytokines are coloured (more colours cannot be told apart).", "warn")
    return None


def ex9_render(color_by, pc_x, pc_y, cytokines, gene):
    g = _pca_common_msgs(color_by, gene, cytokines)
    if color_by == "gene" and g is None:
        return
    if pc_x == pc_y:
        msg("Choose two <b>different</b> PCs for the two axes.", "info"); return
    P = PCA_ALL
    fig = plt.figure(figsize=(13, 5.8))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.45, 1], height_ratios=[1, 1.15], hspace=0.55, wspace=0.28)
    ax = fig.add_subplot(gs[:, 0])
    pca_scatter(ax, P, color_by, (pc_x, pc_y), cytokines, g)
    ax.set_title(f"All {adata.n_obs:,} samples · PCA of the {N_HVG:,} most variable genes")
    plot_scree(fig.add_subplot(gs[0, 1]), P, (pc_x, pc_y))
    plot_r2(fig.add_subplot(gs[1, 1]), P, (pc_x, pc_y), ["cell type", "donor", "cytokine", "control vs treated", "library size", "number of cells"])
    show_fig(fig)


def exercise_pca_all():
    cyts = [c for c in adata.obs["cytokine"].cat.categories if c != "PBS"]
    c = dict(color_by=dropdown(COLOR_CHOICES, "donor", "280px"), pc_x=dropdown(PC_CHOICES, 0, "100px"), pc_y=dropdown(PC_CHOICES, 1, "100px"),
             cytokines=multi_select(cyts, pick_default(cyts, ["IFN-gamma", "IL-4", "TNF-alpha", "IL-10", "IL-6", "GM-CSF"]), rows=7, width="170px"),
             gene=gene_combo("LYZ", "160px"))
    f_cyt = widgets.VBox([field("Cytokines to highlight", c["cytokines"], "up to 8; PBS is always black"), selection_summary(c["cytokines"], cyt_label, 4)])
    f_gene = field("Gene", c["gene"], "type a symbol")

    def _toggle(*_):
        f_cyt.layout.display = None if c["color_by"].value == "cytokine" else "none"
        f_gene.layout.display = None if c["color_by"].value == "gene" else "none"
    c["color_by"].observe(_toggle, "value"); _toggle()
    ex = Exercise(c, ex9_render, [[field("Colour the samples by…", c["color_by"]), field("x axis", c["pc_x"]), field("y axis", c["pc_y"]), f_cyt, f_gene]], title="PCA of the complete dataset")
    ex.show()
    return ex


# ── Exercise 10: one cell type at a time ──────────────────────────────────────────────────────────
COLOR_CHOICES_CT = [("Cytokine (highlight a few)", "cytokine"), (disp("Donor"), "donor"), ("Control vs treated", "treatment"), ("Library size", "library_size"), ("Expression of a gene", "gene")]


def plot_loadings(ax, P, pc, n=8):
    """The genes with the largest weights (loadings) of one PC."""
    s = pd.Series(P["loadings"][:, pc], index=P["hvg_names"]).sort_values()
    top = pd.concat([s.head(n), s.tail(n)])
    ax.barh(top.index, top.values, color=[DOWN_COLOR if v < 0 else UP_COLOR for v in top.values])
    ax.axvline(0, color="k", lw=0.6); ax.tick_params(axis="y", labelsize=8)
    for lab in ax.get_yticklabels():
        lab.set_fontstyle("italic")
    ax.set_xlabel("loading (weight of the gene)"); ax.set_title(f"Genes behind PC{pc + 1}")


def ex10_render(cell_type, color_by, pc_x, pc_y, cytokines, gene, show_loadings):
    g = _pca_common_msgs(color_by, gene, cytokines)
    if color_by == "gene" and g is None:
        return
    if pc_x == pc_y:
        msg("Choose two <b>different</b> PCs for the two axes.", "info"); return
    P = PCA_CT[cell_type]
    if show_loadings:
        fig = plt.figure(figsize=(13.5, 5.4))
        gs = fig.add_gridspec(2, 2, width_ratios=[1.6, 1], hspace=0.55, wspace=0.55)
        ax = fig.add_subplot(gs[:, 0])
        plot_loadings(fig.add_subplot(gs[0, 1]), P, pc_x)
        plot_loadings(fig.add_subplot(gs[1, 1]), P, pc_y)
    else:
        fig, ax = plt.subplots(figsize=(8.6, 5.2))
    pca_scatter(ax, P, color_by, (pc_x, pc_y), cytokines, g, s=22)
    ax.set_title(f"{CT_LABEL[cell_type]} only ({P['obs'].shape[0]:,} samples): PCA of its {N_HVG:,} most variable genes")
    show_fig(fig)
    R2 = P["r2"]
    rows = {f"PC{pc + 1}": {"variance": f"{P['var'][pc]:.1f} %", "R² donor": f"{R2.loc[f'PC{pc + 1}', 'donor']:.2f}", "R² cytokine": f"{R2.loc[f'PC{pc + 1}', 'cytokine']:.2f}",
                              "R² library size": f"{R2.loc[f'PC{pc + 1}', 'library size']:.2f}"} for pc in (pc_x, pc_y)}
    display(HTML('<div class="cx-label">What do these two PCs capture in this cell type?</div>' + html_table(pd.DataFrame(rows).T, index_label="PC")))


def exercise_pca_celltype():
    cyts = [c for c in adata.obs["cytokine"].cat.categories if c != "PBS"]
    c = dict(cell_type=dropdown([(CT_LABEL[x], x) for x in CELL_TYPES], "CD14_Mono", "190px"), color_by=dropdown(COLOR_CHOICES_CT, "cytokine", "250px"),
             pc_x=dropdown(PC_CHOICES, 0, "100px"), pc_y=dropdown(PC_CHOICES, 3, "100px"),
             cytokines=multi_select(cyts, pick_default(cyts, ["IFN-beta", "IFN-gamma", "IL-4", "IL-13", "GM-CSF", "IL-10"]), rows=7, width="170px"),
             gene=gene_combo("UTY", "160px"), show_loadings=checkbox("show the genes behind each axis", True, "250px"))
    f_cyt = widgets.VBox([field("Cytokines to highlight", c["cytokines"], "up to 8; PBS is always black"), selection_summary(c["cytokines"], cyt_label, 4)])
    f_gene = field("Gene", c["gene"], "type a symbol")

    def _toggle(*_):
        f_cyt.layout.display = None if c["color_by"].value == "cytokine" else "none"
        f_gene.layout.display = None if c["color_by"].value == "gene" else "none"
    c["color_by"].observe(_toggle, "value"); _toggle()
    ex = Exercise(c, ex10_render, [[field("Cell type", c["cell_type"]), field("Colour the samples by…", c["color_by"]), field("x axis", c["pc_x"]), field("y axis", c["pc_y"]), f_cyt, f_gene],
                                   [c["show_loadings"]]], title="PCA within one cell type")
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 11: volcano plots of the pre-computed DESeq2 results (cytokine vs PBS, one cell type)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
PADJ_OPTIONS = [0.001, 0.01, 0.05, 0.1, 0.2]
try:
    from adjustText import adjust_text
    HAS_ADJUSTTEXT = True
except Exception:
    HAS_ADJUSTTEXT = False


def de_summary(res, padj_thr, lfc_thr):
    """Add the up / down / ns call and the -log10 adjusted p-value to a DE table."""
    d = res.copy()
    d["direction"] = classify_de(d, padj_thr, lfc_thr)
    d["nlp"] = d["neglog10_padj"]
    d["score"] = d["nlp"] * d["log2fc"].abs()                                    # evidence × effect size: used to pick the genes to label
    return d


def mean_expression(gene_list, cell_type, cytokine):
    """Mean log2(CPM + 1) of each gene in the 12 PBS and the 12 cytokine samples of one cell type."""
    r0 = adata.obs_names.get_indexer([f"{d}|{cell_type}|PBS" for d in DONORS])
    r1 = adata.obs_names.get_indexer([f"{d}|{cell_type}|{cytokine}" for d in DONORS])
    gi = [gene_index(g) for g in gene_list]
    L = adata.layers["log2cpm"]
    return L[np.ix_(r0, gi)].mean(axis=0), L[np.ix_(r1, gi)].mean(axis=0)


def axis_caps(x, y):
    """A few genes have extreme values (e.g. log2FC = 12 when a gene is off in PBS): they are drawn on the edge instead of squeezing the plot."""
    xq, yq = np.quantile(np.abs(x), 0.999), np.quantile(y, 0.999)
    return min(np.abs(x).max(), max(2.0 * xq, 3.0)) * 1.04, min(y.max(), max(1.8 * yq, 10.0)) * 1.04


def draw_volcano(ax, d, padj_thr, lfc_thr, n_labels, highlight=(), title=""):
    t = d[d["tested"]].copy()
    xcap, ycap = axis_caps(t["log2fc"].values, t["nlp"].values)
    t["x"], t["y"] = t["log2fc"].clip(-xcap, xcap), t["nlp"].clip(upper=ycap)
    beyond = (t["x"] != t["log2fc"]) | (t["y"] != t["nlp"])
    for k, col, size, alpha in (("ns", NS_COLOR, 6, 0.55), ("down", DOWN_COLOR, 12, 0.8), ("up", UP_COLOR, 12, 0.8)):
        sub = t[(t["direction"] == k) & ~beyond]
        ax.scatter(sub["x"], sub["y"], s=size, color=col, alpha=alpha, linewidths=0, rasterized=True, zorder=2)
        sub = t[(t["direction"] == k) & beyond]
        ax.scatter(sub["x"], sub["y"], s=40, color=col, marker="^", linewidths=0.5, edgecolors="k", zorder=3)
    for x in (-lfc_thr, lfc_thr):
        ax.axvline(x, color="#999", lw=0.8, ls="--", zorder=1)
    ax.axhline(-np.log10(padj_thr), color="#999", lw=0.8, ls="--", zorder=1)
    ax.set_xlim(-xcap * 1.04, xcap * 1.04); ax.set_ylim(-ycap * 0.03, ycap * 1.1)
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    sig = t[t["direction"] != "ns"]
    hl = [g for g in highlight if g in t.index]
    texts = []
    if n_labels > 0 and len(sig):                                               # the strongest DEGs, half up and half down
        for k in ("up", "down"):
            top = sig[sig["direction"] == k].nlargest(max(1, n_labels // 2), "score")
            for g, r in top.iterrows():
                if g not in hl and not (r["nlp"] > y0 + 0.85 * (y1 - y0) and (r["log2fc"] < x0 + 0.25 * (x1 - x0) or r["log2fc"] > x1 - 0.25 * (x1 - x0))):
                    texts.append(ax.text(r["x"], r["y"], g, fontsize=7.5, fontstyle="italic", zorder=5))
    if hl:                                                                       # the genes chosen by the student
        r = t.loc[hl]
        ax.scatter(r["x"], r["y"], s=70, facecolors="none", edgecolors="k", linewidths=1.6, zorder=6)
        for g, rr in r.iterrows():
            texts.append(ax.text(rr["x"], rr["y"], g, fontsize=9.5, fontweight="bold", fontstyle="italic", zorder=7,
                                 bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="k", lw=0.6, alpha=0.9)))
    if HAS_ADJUSTTEXT and texts:
        try:
            adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="#888", lw=0.5), expand=(1.2, 1.4), max_move=(30, 30))
        except Exception:
            pass
    n_up, n_down = int((t["direction"] == "up").sum()), int((t["direction"] == "down").sum())
    ax.annotate(f"↑ {n_up:,} up", xy=(0.98, 0.98), xycoords="axes fraction", ha="right", va="top", color=UP_COLOR, fontsize=11, fontweight="bold")
    ax.annotate(f"↓ {n_down:,} down", xy=(0.02, 0.98), xycoords="axes fraction", ha="left", va="top", color=DOWN_COLOR, fontsize=11, fontweight="bold")
    foot = f"{len(t):,} genes tested · {len(t) - n_up - n_down:,} not significant" + (f"   ·   ▲ {int(beyond.sum())} beyond the axes (see the tables)" if beyond.any() else "")
    ax.text(0.5, 1.005, foot, transform=ax.transAxes, ha="center", va="bottom", color="#777", fontsize=8.5)
    ax.set_xlabel("log₂ fold change  (cytokine / PBS)"); ax.set_ylabel("−log₁₀ adjusted p-value")
    ax.set_title(title, pad=16)
    return n_up, n_down


def _status(r, padj_thr, lfc_thr):
    if not r["tested"]:
        return "not analysed (outside the immune-gene list, or too few counts)"
    return {"up": '<span class="cx-up">↑ up-regulated</span>', "down": '<span class="cx-down">↓ down-regulated</span>', "ns": "not significant"}[r["direction"]]


def gene_stats_table(d, genes, cell_type, cytokine, padj_thr, lfc_thr):
    m0, m1 = mean_expression(genes, cell_type, cytokine)
    rows = []
    for g, a, b in zip(genes, m0, m1):
        r = d.loc[g]
        rows.append({"gene": g, "log₂ fold change": r["log2fc"] if r["tested"] else np.nan, "fold change": (2 ** r["log2fc"]) if r["tested"] else np.nan,
                     "adjusted p-value": f"{r['padj']:.1e}" if r["tested"] else "—", "mean in PBS": a, f"mean with {cyt_label(cytokine)}": b, "verdict": _status(r, padj_thr, lfc_thr)})
    df = pd.DataFrame(rows).set_index("gene")
    return html_table(df, float_fmt="{:,.2f}", left_cols=["verdict"], index_label=disp("gene"))


def top_table(d, direction, n=10):
    t = d[(d["direction"] == direction)].nlargest(n, "score")
    df = pd.DataFrame({"log₂ fold change": t["log2fc"], "fold change": 2.0 ** t["log2fc"], "adjusted p-value": t["padj"].map(lambda x: f"{x:.1e}")})
    return html_table(df, float_fmt="{:,.2f}", index_label=disp("gene"))


def ex11_render(cell_type, cytokine, padj_thr, lfc_thr, n_labels, genes_text):
    if cytokine not in DE.cytokines and DE.cached:
        msg("Choose a cytokine from the list.", "info"); return
    highlight, unknown = parse_genes(genes_text, 25)
    unknown_genes_message(unknown)
    if not DE.cached and (cell_type, cytokine) not in DE._memo:
        msg("⏳ The pre-computed results were not found: running DESeq2 for this comparison (≈ 15 seconds)…", "info")
    res = DE.get(cell_type, cytokine)
    d = de_summary(res, padj_thr, lfc_thr)
    fig, ax = plt.subplots(figsize=(8.6, 6.4))
    draw_volcano(ax, d, padj_thr, lfc_thr, n_labels, highlight, f"{cyt_label(cytokine)} vs PBS · {CT_LABEL[cell_type]}")
    fig.tight_layout()
    show_fig(fig)
    if highlight:
        display(HTML(f'<div class="cx-label">Your genes</div>' + gene_stats_table(d, highlight, cell_type, cytokine, padj_thr, lfc_thr)))
    display(HTML('<div style="display:flex; flex-wrap:wrap; gap:28px"><div><div class="cx-label">Top 10 up-regulated genes <span class="cx-hint">(strongest evidence × effect)</span></div>'
                 + top_table(d, "up") + '</div><div><div class="cx-label">Top 10 down-regulated genes</div>' + top_table(d, "down") + '</div></div>'))


def exercise_volcano():
    cyts = [c for c in adata.obs["cytokine"].cat.categories if c != "PBS"]
    c = dict(cell_type=dropdown([(CT_LABEL[x], x) for x in CELL_TYPES], "CD14_Mono", "190px"), cytokine=dropdown([(cyt_label(x), x) for x in cyts], "IFN-gamma", "190px"),
             padj_thr=widgets.SelectionSlider(options=PADJ_OPTIONS, value=PADJ_DEFAULT, continuous_update=False, layout=_lay("300px"), style=STYLE),
             lfc_thr=float_slider(0, 3, LFC_DEFAULT, 0.25, "270px", ".2f"), n_labels=int_slider(0, 30, 12, 2, "230px"), genes_text=None)
    gtext, gwidget = gene_picker("CXCL9, CXCL10, IDO1, STAT1, IRF1", "300px", "e.g. CXCL9, IDO1", label="Search and highlight genes")
    c["genes_text"] = gtext
    rows = [[field("Cell type", c["cell_type"]), field("Cytokine (vs PBS)", c["cytokine"]), field("Significance: adjusted p-value below…", c["padj_thr"]),
             field("Effect size: |log₂ fold change| above…", c["lfc_thr"], "0.5 ≈ 1.4× · 1 = 2× · 2 = 4×"), field("Gene labels", c["n_labels"], "strongest genes")], [gwidget]]
    ex = Exercise(c, ex11_render, rows, title="Which genes change with a cytokine?")
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 12: comparing two DE signatures (correlation of log2 fold changes)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
GENE_SUBSETS = [("all genes tested in both comparisons", "all"), ("genes significant in A or B", "either"), ("genes significant in both A and B", "both")]
EXAMPLES12 = [("— choose an example —", ""),
              ("IL-4 vs IL-13 in monocytes: almost the same response", ("CD14_Mono", "IL-4", "CD14_Mono", "IL-13")),
              ("IL-4 vs IL-13 in CD4 T cells: much less similar", ("CD4_T_cell", "IL-4", "CD4_T_cell", "IL-13")),
              ("IFN-β vs IFN-γ in monocytes: two interferons", ("CD14_Mono", "IFN-beta", "CD14_Mono", "IFN-gamma")),
              ("IFN-γ vs IL-4 in monocytes: opposite programmes", ("CD14_Mono", "IFN-gamma", "CD14_Mono", "IL-4")),
              ("TNF-α in monocytes vs in CD4 T cells: same stimulus, two cell types", ("CD14_Mono", "TNF-alpha", "CD4_T_cell", "TNF-alpha")),
              ("IL-6 in monocytes vs in CD4 T cells: same stimulus, two cell types", ("CD14_Mono", "IL-6", "CD4_T_cell", "IL-6")),
              ("TNF-α vs IL-6 in monocytes", ("CD14_Mono", "TNF-alpha", "CD14_Mono", "IL-6")),
              ("IL-2 vs IL-15 in NK cells: the same receptor", ("NK_CD56low", "IL-2", "NK_CD56low", "IL-15"))]
COL_A, COL_B, COL_BOTH = BLUE, ORANGE, PURPLE


def compare_tables(da, db, subset, padj_thr, lfc_thr):
    """Merge two DE tables on the genes tested in both and compute the statistics of the comparison."""
    ok = da["tested"] & db["tested"]
    m = pd.DataFrame({"a": da.loc[ok, "log2fc"], "b": db.loc[ok, "log2fc"], "dir_a": da.loc[ok, "direction"], "dir_b": db.loc[ok, "direction"]})
    sa, sb = m["dir_a"] != "ns", m["dir_b"] != "ns"
    m["cls"] = np.select([sa & sb, sa, sb], ["both", "A", "B"], "none")
    use = {"all": np.ones(len(m), bool), "either": (sa | sb).values, "both": (sa & sb).values}[subset]
    return m, use


def correlations(m, use):
    x, y = m.loc[use, "a"].values.astype(float), m.loc[use, "b"].values.astype(float)
    if len(x) < 5 or np.std(x) == 0 or np.std(y) == 0:
        return float("nan"), float("nan"), len(x)
    return float(np.corrcoef(x, y)[0, 1]), float(stats.spearmanr(x, y)[0]), len(x)


def similarity_ranking(ct, cyt, subset_all=True):
    """Pearson correlation of the log2FC of (ct, cyt) with every other cytokine in the same cell type (cache only)."""
    i, j = DE.cell_types.index(ct), DE.cytokines.index(cyt)
    lfc, nlp = DE.arrays["lfc"][i].astype(np.float32), DE.arrays["nlp_adj"][i].astype(np.float32)
    out = {}
    for k, name in enumerate(DE.cytokines):
        if k == j:
            continue
        ok = ~np.isnan(nlp[j]) & ~np.isnan(nlp[k])
        if ok.sum() > 50:
            out[name] = float(np.corrcoef(lfc[j][ok], lfc[k][ok])[0, 1])
    return pd.Series(out).sort_values(ascending=False)


def ex12_render(cell_a, cyt_a, cell_b, cyt_b, subset, padj_thr, lfc_thr, genes_text, label_top):
    if cyt_a == cyt_b and cell_a == cell_b:
        msg("A and B are the <b>same comparison</b>: the correlation is trivially 1. Change the cytokine or the cell type of B.", "info")
    if not DE.cached and any((c, y) not in DE._memo for c, y in ((cell_a, cyt_a), (cell_b, cyt_b))):
        msg("⏳ The pre-computed results were not found: running DESeq2 (≈ 15 seconds per comparison)…", "info")
    da, db = de_summary(DE.get(cell_a, cyt_a), padj_thr, lfc_thr), de_summary(DE.get(cell_b, cyt_b), padj_thr, lfc_thr)
    m, use = compare_tables(da, db, subset, padj_thr, lfc_thr)
    highlight, unknown = parse_genes(genes_text, 25)
    unknown_genes_message(unknown)
    r, rho, n = correlations(m, use)
    na, nb = f"{cyt_label(cyt_a)} · {CT_SHORT[cell_a]}", f"{cyt_label(cyt_b)} · {CT_SHORT[cell_b]}"
    ranking = similarity_ranking(cell_a, cyt_a) if DE.cached else None
    fig = plt.figure(figsize=(13, 6.0))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.35, 1], wspace=0.35)
    ax = fig.add_subplot(gs[0])
    for cls, col, lab, s_ in (("none", NS_COLOR, "neither", 5), ("A", COL_A, "A only", 11), ("B", COL_B, "B only", 11), ("both", COL_BOTH, "both", 13)):
        mm = m[m["cls"] == cls]
        ax.scatter(mm["a"], mm["b"], s=s_, alpha=0.55 if cls == "none" else 0.8, color=col, linewidths=0, label=f"{disp(lab)}  ({len(mm):,})", rasterized=True, zorder=2)
    ax.axhline(0, color="k", lw=0.6, zorder=1); ax.axvline(0, color="k", lw=0.6, zorder=1)
    lim = float(np.nanpercentile(np.abs(np.r_[m["a"].values, m["b"].values]), 99.8)) * 1.1
    ax.plot([-lim, lim], [-lim, lim], color="#999", lw=0.9, ls="--", zorder=1)
    sel = m[use]
    if len(sel) > 5 and np.std(sel["a"]) > 0:
        slope, icpt = np.polyfit(sel["a"].astype(float), sel["b"].astype(float), 1)
        xs = np.array([-lim, lim]); ax.plot(xs, icpt + slope * xs, color="k", lw=1.8, zorder=4)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    texts = []
    if label_top and len(sel):
        mn = np.minimum(sel["a"].abs(), sel["b"].abs())
        same, opp = sel[np.sign(sel["a"]) == np.sign(sel["b"])], sel[np.sign(sel["a"]) != np.sign(sel["b"])]
        for part, k in ((same, 5), (opp, 3)):
            for g in mn[part.index].nlargest(k).index:
                if g not in highlight:
                    texts.append(ax.text(m.loc[g, "a"], m.loc[g, "b"], g, fontsize=7.5, fontstyle="italic", zorder=5))
    hl = [g for g in highlight if g in m.index]
    if hl:
        ax.scatter(m.loc[hl, "a"], m.loc[hl, "b"], s=70, facecolors="none", edgecolors="k", linewidths=1.6, zorder=6)
        for g in hl:
            texts.append(ax.text(m.loc[g, "a"], m.loc[g, "b"], g, fontsize=9.5, fontweight="bold", fontstyle="italic", zorder=7, bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="k", lw=0.6, alpha=0.9)))
    if HAS_ADJUSTTEXT and texts:
        try:
            adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="#888", lw=0.5), expand=(1.2, 1.4), max_move=(30, 30))
        except Exception:
            pass
    ax.legend(loc="lower right", markerscale=2.2, fontsize=8.5, title="significant in…", title_fontsize=8.5)
    ax.set_xlabel(f"log₂ fold change of A:  {cyt_label(cyt_a)} vs PBS in {CT_LABEL[cell_a]}"); ax.set_ylabel(f"log₂ fold change of B:  {cyt_label(cyt_b)} vs PBS in {CT_LABEL[cell_b]}")
    ax.set_title(f"Pearson r = {r:.2f}   ·   Spearman ρ = {rho:.2f}   ·   {n:,} genes", fontsize=12)
    ax2 = fig.add_subplot(gs[1])
    if ranking is not None and cell_a == cell_b and cyt_b in ranking.index:
        sel_r = pd.concat([ranking.head(8), ranking.tail(4)])
        if cyt_b not in sel_r.index:
            sel_r = pd.concat([ranking.head(8), ranking.loc[[cyt_b]], ranking.tail(3)])
        cols = [COL_B if k == cyt_b else (UP_COLOR if v > 0 else DOWN_COLOR) for k, v in sel_r.items()]
        ax2.barh([cyt_label(k) for k in sel_r.index], sel_r.values, color=cols)
        ax2.axvline(0, color="k", lw=0.6); ax2.invert_yaxis()
        ax2.set_xlabel("correlation of the log₂ fold changes with A"); ax2.set_title(f"In {CT_LABEL[cell_a]}, who responds like {cyt_label(cyt_a)}?\n(B = {cyt_label(cyt_b)} in orange; rank {int((ranking > ranking[cyt_b]).sum()) + 1} of {len(ranking)})", fontsize=10)
        ax2.tick_params(axis="y", labelsize=8.5)
    elif ranking is not None:
        sel_r = pd.concat([ranking.head(8), ranking.tail(4)])
        ax2.barh([cyt_label(k) for k in sel_r.index], sel_r.values, color=[UP_COLOR if v > 0 else DOWN_COLOR for v in sel_r.values])
        ax2.axvline(0, color="k", lw=0.6); ax2.invert_yaxis(); ax2.tick_params(axis="y", labelsize=8.5)
        ax2.set_xlabel("correlation of the log₂ fold changes with A"); ax2.set_title(f"In {CT_LABEL[cell_a]}, who responds like {cyt_label(cyt_a)}?", fontsize=10)
    else:
        ax2.axis("off")
    fig.tight_layout()
    show_fig(fig)
    # which genes explain the (dis)similarity?
    mn = np.minimum(sel["a"].abs(), sel["b"].abs()) if len(sel) else pd.Series(dtype=float)
    def gene_tab(part, k=8):
        top = mn[part.index].nlargest(k).index
        df = pd.DataFrame({"log₂FC in A": m.loc[top, "a"], "log₂FC in B": m.loc[top, "b"]})
        return html_table(df, index_label="gene")
    if len(sel):
        same_up = sel[(sel["a"] > 0) & (sel["b"] > 0)]; same_dn = sel[(sel["a"] < 0) & (sel["b"] < 0)]; opp = sel[np.sign(sel["a"]) != np.sign(sel["b"])]
        display(HTML(
            f'<div class="cx-info">Among the {n:,} genes used: <b>{len(same_up):,}</b> go up in both, <b>{len(same_dn):,}</b> go down in both, '
            f'<b>{len(opp):,}</b> change in opposite directions. Pearson r = <b>{r:.2f}</b> (Spearman ρ = {rho:.2f}).</div>'
            '<div style="display:flex; flex-wrap:wrap; gap:28px">'
            f'<div><div class="cx-label">Up in both (largest shared change)</div>{gene_tab(same_up)}</div>'
            f'<div><div class="cx-label">Down in both</div>{gene_tab(same_dn)}</div>'
            f'<div><div class="cx-label">Opposite directions</div>{gene_tab(opp, 6)}</div></div>'))
    else:
        msg("No genes pass the selected filter: try <i>all genes tested in both comparisons</i>, or relax the thresholds.", "info")
    if hl:
        df = pd.DataFrame({"log₂FC in A": m.loc[hl, "a"], "adj. p in A": [f"{da.loc[g, 'padj']:.1e}" for g in hl], "log₂FC in B": m.loc[hl, "b"], "adj. p in B": [f"{db.loc[g, 'padj']:.1e}" for g in hl]}, index=hl)
        display(HTML('<div class="cx-label" style="margin-top:8px">Your genes</div>' + html_table(df, index_label="gene")))
    missing = [g for g in highlight if g not in m.index]
    if missing:
        msg("Not analysed in both comparisons (outside the immune-gene list, or too few counts): " + ", ".join(missing), "warn")


def exercise_compare():
    cyts = [c for c in adata.obs["cytokine"].cat.categories if c != "PBS"]
    cts = [(CT_LABEL[x], x) for x in CELL_TYPES]
    cyt_opts = [(cyt_label(x), x) for x in cyts]
    c = dict(cell_a=dropdown(cts, "CD14_Mono", "170px"), cyt_a=dropdown(cyt_opts, "IL-4", "170px"), cell_b=dropdown(cts, "CD14_Mono", "170px"), cyt_b=dropdown(cyt_opts, "IL-13", "170px"),
             subset=dropdown(GENE_SUBSETS, "all", "290px"),
             padj_thr=widgets.SelectionSlider(options=PADJ_OPTIONS, value=PADJ_DEFAULT, continuous_update=False, layout=_lay("260px"), style=STYLE),
             lfc_thr=float_slider(0, 3, LFC_DEFAULT, 0.25, "230px", ".2f"), genes_text=None, label_top=checkbox("label the genes that drive the pattern", True, "300px"))
    gtext, gwidget = gene_picker("", "280px", "e.g. STAT1, CCL17", with_sets=False, label="Highlight genes (optional)")
    c["genes_text"] = gtext
    example = dropdown(EXAMPLES12, "", "480px")
    ex = None

    def _load(change):
        v = change["new"]
        if v:
            ex.set_values(cell_a=v[0], cyt_a=v[1], cell_b=v[2], cyt_b=v[3])
            example.value = ""
    example.observe(_load, "value")
    rows = [[field("Load an example", example, "…or choose A and B yourself below")],
            [widgets.VBox([widgets.HTML('<div class="cx-label" style="color:%s">COMPARISON A</div>' % COL_A), widgets.HBox([field("Cell type", c["cell_a"]), field("Cytokine vs PBS", c["cyt_a"])])]),
             widgets.VBox([widgets.HTML('<div class="cx-label" style="color:%s">COMPARISON B</div>' % COL_B), widgets.HBox([field("Cell type", c["cell_b"]), field("Cytokine vs PBS", c["cyt_b"])])])],
            [field("Genes used for the correlation", c["subset"]), field("“Significant” = adjusted p below…", c["padj_thr"]), field("…and |log₂FC| above", c["lfc_thr"]), gwidget, c["label_top"]]]
    ex = Exercise(c, ex12_render, rows, title="Do two stimuli trigger the same genes?")
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  90_init.py: everything slow happens here, ONCE (load the data, normalise, PCA, load the DESeq2 results)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
GENESETS_WANTED = {
    "Monocyte markers": ["LYZ", "CD14", "S100A9", "FCN1"],
    "T-cell markers": ["CD3E", "CD3D"],
    "CD8 T-cell markers": ["CD8A", "CD8B"],
    "NK-cell markers": ["NKG7", "GNLY", "KLRF1"],
    "IFN-γ / STAT1 response": ["STAT1", "IRF1", "GBP1", "IDO1", "CXCL9", "CXCL10"],
    "Type I interferon response": ["ISG15", "IFIT1", "IFIT3", "MX1", "OAS1"],
    "IL-4 / STAT6 response": ["ALOX15", "CCL17", "FCER2"],
    "IL-1β / NF-κB response": ["IL6", "CXCL8", "CCL2"],
    "IL-2 / STAT5 response": ["IL2RA", "GZMB", "CISH"],
    "Housekeeping genes": ["ACTB", "GAPDH", "B2M", "TMSB4X", "UBC"],
}

adata = None          # the AnnData object of the practical (X = raw counts; layers: cpm, log2cpm)
EXPRESSED = None      # True for the genes that pass the expression filter (≥ 10 counts in ≥ 12 samples)
PCA_ALL, PCA_CT, DE = None, {}, None


def init_practical(h5ad_path=None, de_cache_path=None, verbose=True, with_de=True):
    """Load the data and pre-compute everything the exercises need. Takes ~30 s once the files are on disk."""
    global adata, EXPRESSED, PCA_ALL, PCA_CT, DE, GENE_NAMES, GENE_LOOKUP, GENESETS
    t0 = time.time()
    say = (lambda s: print(s, flush=True)) if verbose else (lambda s: None)
    path = get_h5ad(h5ad_path, verbose)
    say("📖 Reading and tidying the data …")
    adata = load_adata(path)
    EXPRESSED = is_expressed(adata.X)
    GENE_NAMES = list(adata.var_names)
    GENE_LOOKUP = {}
    for g in GENE_NAMES:
        GENE_LOOKUP.setdefault(g.upper(), g)
    GENESETS = {k: [g for g in v if g in GENE_LOOKUP.values()] for k, v in GENESETS_WANTED.items()}
    say("🧮 Running the PCAs (all samples, and one per cell type) …")
    PCA_ALL = make_pca(adata, expressed=EXPRESSED)
    PCA_CT = {ct: make_pca(adata, mask=(adata.obs["cell_type"] == ct).values, expressed=EXPRESSED) for ct in CELL_TYPES}
    if with_de:
        say("📦 Loading the differential-expression results …")
        DE = DEResults(adata, de_cache_path or get_de_cache(verbose), verbose)
    say(f"✅ Ready in {time.time() - t0:.0f} s: {adata.n_obs:,} samples × {adata.n_vars:,} genes. Scroll down and start with Exercise 1.")
    return adata


# ── Load the data and pre-compute everything ──────────────────────────────────────────────────────
init_practical()


---
# Part A · Python warm-up
Two short exercises that need no data: what a **variable** is, and how to **look at numbers** with plots.

## Exercise 1 · Variables: giving names to values
⏱ *5 min*

A **variable** is a **name** that stores a **value**, like a labelled tube in the freezer. You create one with `=`: the name on the left, the value on the right.
Python needs to know *what kind* of value it is (its **type**): **text** (`str`, always between quotes), **whole numbers** (`int`), **decimal numbers** (`float`), **True/False** (`bool`) or a **list** of values.
`print(...)` shows a value on the screen, and variables can be combined to create new ones.

**What to do:** change the controls and watch how the Python code, the printed output and the table of variables change.

In [ ]:
#@title ▶ Exercise 1 · Variables { display-mode: "form" }
try:
    exercise_variables()
except NameError:
    print('⚠️ Run the Setup cell at the top of the notebook first (▶), wait for ✅ Ready, and then run this cell again.')


#### ❓ Questions
1. Change `message` to your own name. Which lines of the code panel and of the output change?
2. Move the `n_donors` slider. Which printed lines change and which do not? Why?
3. In the table of variables, which type is `library_size`? Why is it not a whole number?

<details>
<summary>💡 Answers</summary>

1. The line `message = '...'` and the first printed line. Text values are written **between quotes**; the variable keeps whatever you type.
2. Only the lines that *use* `n_donors`: `Donors: …` and `… was added to … donors`. A change in a variable only affects the places where its name is used.
3. It is a `float` (decimal): `library_size = n_cells * umis_per_cell` multiplies a whole number by a decimal number, and the result is a decimal. If both were whole numbers (`int`), the result would be an `int`. (`library_size`, the total number of counts of a sample, is a concept we will meet again in Exercise 3.)

</details>

## Exercise 2 · Looking at numbers: basic plots
⏱ *7 min*

A plot turns many numbers into one picture. Different plots answer different questions:

| Plot | Answers the question… |
|---|---|
| 📊 **Histogram** | *How are the values distributed?* (how many donors have each value) |
| 📦 **Box plot** | *Where is the middle and how spread out are the values?* Line = median, box = the middle 50 %, whiskers = the range |
| 📶 **Bar chart** | *What is the average of each group?* (it hides the individual values!) |
| 🔵 **Scatter plot** | *Are two measurements related?* One point per donor |
| 📈 **Line plot** | *How does a value change over time?* |

**What to do:** the data are **simulated**: one gene measured in several donors, without (control) and with (treated) a cytokine. Pick a plot type and play with the numbers.

In [ ]:
#@title ▶ Exercise 2 · Basic plots { display-mode: "form" }
try:
    exercise_plotting()
except NameError:
    print('⚠️ Run the Setup cell at the top of the notebook first (▶), wait for ✅ Ready, and then run this cell again.')


#### ❓ Questions
1. In the **histogram**, set the effect of the cytokine to 0 and then to +2. What moves? What happens if you also increase the variability between donors (SD)?
2. Use few donors (5–10) and press **🎲 new random draw** several times (histogram or box plot). Do the two groups always look the same? What does this tell you about experiments with few donors?
3. In the **scatter plot**, use an effect of +1 and compare a similarity of 0.1 with one of 0.95. When is the effect easier to see? Why would it help to compare every donor with *itself*?

<details>
<summary>💡 Answers</summary>

1. With effect 0 the two histograms overlap completely; with +2 the treated one shifts to the right by 2 units (2 units of log₂ = **4 times** more expression). A larger SD makes both histograms wider, so they overlap more even if the effect is the same: the effect is harder to see when donors differ a lot.
2. No: with few donors, chance alone changes the means and the shapes, and you can see "differences" (or miss real ones) by bad luck. More donors give stable pictures. This is why experiments need replicates.
3. With similarity 0.95 all the points sit in a narrow band **above** the dashed line (`y = x`): *every* donor goes up by about the same amount, even though donors start from very different levels. With 0.1 the cloud is round and the shift is hidden by the differences between donors. Comparing each donor with itself (a **paired** design, Exercises 11–12) removes the donor-to-donor differences.

</details>

---
# Part B · Meet the data

## Exercise 3 · The AnnData object: samples, genes and their metadata
⏱ *10 min*

Our data live in an **AnnData** object called `adata`, which keeps several tables **aligned**:

| Component | What it holds | Shape |
|---|---|---|
| `X` | the **numbers**: raw counts (how many mRNA molecules of each gene were detected) | samples × genes |
| `obs` | information about the **samples** (rows of `X`): donor, cytokine, cell type… | one row per sample |
| `var` | information about the **genes** (columns of `X`) | one row per gene |
| `layers` | other versions of `X` with the same shape (here, *normalized* versions we will build in Exercises 5–6) | samples × genes |

The complete object has **4,224 samples × 18,120 genes**: 12 donors × 4 cell types × 88 conditions (87 cytokines + PBS).
A **sample** is a pseudobulk: all the cells of one donor, one cell type and one condition, added together. Its name is `donor|cell type|cytokine`.
Selecting works like a table: `adata[samples, genes]`, **samples first, genes second**.

**What to do:** choose samples and genes (the defaults are the 4 PBS controls of Donor 1, plus its monocytes treated with IFN-γ). The **subset** you build is shown in the three tabs, and its place in the full matrix on the right.

In [ ]:
#@title ▶ Exercise 3 · The AnnData object { display-mode: "form" }
try:
    exercise_anndata()
except NameError:
    print('⚠️ Run the Setup cell at the top of the notebook first (▶), wait for ✅ Ready, and then run this cell again.')


#### ❓ Questions
1. In the default subset, which sample has the highest count of `LYZ`, of `CD3E` and of `NKG7`? Does it match what you know about these genes?
2. Open the **obs** tab: what do `psbulk_n_cells` and `library_size` tell you? Compare the two monocyte samples (PBS and IFN-γ).
3. `STAT1` is switched on by IFN-γ. Compare its count in the two monocyte samples. Is it higher with IFN-γ? Is that what you expected?

<details>
<summary>💡 Answers</summary>

1. `LYZ` is highest in the **monocytes** (911,945 counts), `CD3E` in the **CD4 T cells** (20,361; CD8 T cells are second with 3,100) and `NKG7` in the **NK cells** (2,392; then CD8 T cells, which are also cytotoxic): they are cell-type markers.
2. `psbulk_n_cells` is the **number of cells that were added up** into the sample, and `library_size` the **total number of counts** (of all genes). The PBS monocyte sample contains 17,373 cells and 147.9 million counts; the IFN-γ one only 2,556 cells and 21.4 million counts: about **7 times less material**. (The PBS control is the sum of 6 wells.)
3. **No: it is ~9 times *lower* with IFN-γ** (73,853 vs 639,289), although IFN-γ induces `STAT1`. The raw count mixes **biology** (is the gene switched on?) with the **amount of material** (how many cells, how deep). That is the problem we solve in Part C.

</details>

## Exercise 4 · Distributions of raw counts: boxplots
⏱ *8 min*

A **box plot** summarizes many values: the **line** is the median, the **box** contains the middle 50 % of the values and the **whiskers** show the range. Each **dot** is one **pseudobulk sample**
(a donor with one cytokine in one cell type); the *box* is therefore the distribution of the gene across all samples of a group.
The right-hand panel shows the **library size** (total counts) of the same samples.

**What to do:** the default shows four genes with different roles in the four cell types (all 88 conditions, 1,056 samples per cell type). Change the genes, the groups or the cell types.
Try "control (PBS) vs treated" to compare the two kinds of samples.

In [ ]:
#@title ▶ Exercise 4 · Boxplots of raw counts { display-mode: "form" }
try:
    exercise_boxplots()
except NameError:
    print('⚠️ Run the Setup cell at the top of the notebook first (▶), wait for ✅ Ready, and then run this cell again.')


#### ❓ Questions
1. In the default plot, how many orders of magnitude separate the most and the least abundant gene in monocytes? Do `LYZ`, `CD3E` and `NKG7` behave as cell-type markers?
2. `ACTB` is a housekeeping gene (always on), yet its counts differ between cell types. Compare with the right panel: is there a link?
3. Group the boxes by **control (PBS) vs treated**. What do you see for *every* gene? Why?

<details>
<summary>💡 Answers</summary>

1. In monocytes the median raw count goes from 4 (`NKG7`) to 25,725 (`STAT1`): **almost 4 orders of magnitude**. `LYZ` is highest in monocytes (median 30,058 vs ≤ 297 in the other cells), `CD3E` in CD4 and CD8 T cells (1,262 and 663) and `NKG7` in NK cells (134; then CD8 T cells, 65).
2. Yes: the **library size** differs between cell types (medians: NK 1.2 M, CD8 T 3.8 M, monocytes 7.6 M, CD4 T 8.7 M), and `ACTB` counts follow it largely (NK lowest, CD8 T next). Monocytes are bigger cells with more mRNA, so they have even more `ACTB` per count. Counts depend on how much material each cell type contributes.
3. The **PBS controls are higher for every gene**, including `ACTB`: their library size is ~5 times larger (median 23 M vs 4.4 M) because each PBS sample is the sum of 6 wells. This difference has nothing to do with the biology of each gene, it is the amount of material, which we correct in the next exercise.

</details>

---
# Part C · Normalization: making samples comparable

## Exercise 5 · Raw counts vs CPM: why do we normalize?
⏱ *10 min*

A raw count depends on **two things**: how much the gene is expressed (biology) and how much material we have (how many cells, how deeply sequenced; technical).
The **library size** is the total number of counts of a sample. To remove the effect of depth we express every gene as a fraction of its sample:

**CPM** (counts per million) = count of the gene ÷ library size of the sample × 1,000,000

**What to do:**
* **Toy example** (simulated): the same donor with a deep sample (PBS) and a shallow one (with a cytokine). Only gene B really changes. Move the sliders.
* **Real data, PBS vs cytokine**: one line per donor. The default is the housekeeping gene `ACTB` in monocytes with IFN-γ. Change the gene, the cell type or the cytokine.
* **Real data, all samples**: does the count of the gene follow the library size?

In [ ]:
#@title ▶ Exercise 5 · Raw counts vs CPM { display-mode: "form" }
try:
    exercise_normalization()
except NameError:
    print('⚠️ Run the Setup cell at the top of the notebook first (▶), wait for ✅ Ready, and then run this cell again.')


#### ❓ Questions
1. **Toy example**: with a depth of 20 % and a true fold change of ×4, what do the raw counts say about gene B? And the CPM? What if the depth is 100 %?
2. **Real data**, `ACTB` in monocytes (PBS vs IFN-γ): how do the library size, the raw `ACTB` counts and the `ACTB` CPM change? What do you expect from a housekeeping gene?
3. Change the gene to `IDO1` and then to `STAT1` (both induced by IFN-γ). What do raw counts and CPM say? Then look at the **all samples** view: how does the correlation with the library size change from raw to CPM?

<details>
<summary>💡 Answers</summary>

1. **Raw counts: gene B looks slightly *down* (×0.78)** and every other gene looks ~5 times lower, simply because sample 2 is 5 times shallower. **CPM: genes A, C and D are unchanged (×0.97) and gene B is ×3.9**, close to the true ×4. With a depth of 100 % both representations agree: normalization only matters when depths differ.
2. The library size is **6 times lower** with IFN-γ (×0.16) and the raw `ACTB` count follows it (×0.16); after CPM `ACTB` is **unchanged (×0.95)**, as expected for a housekeeping gene. In the all-samples view the correlation of `ACTB` with the library size is **r = +0.84 in raw counts and −0.25 in CPM**: the depth effect is gone.
3. `IDO1`: raw counts say **no change (×1.1)**, CPM says **×6.7 up**. `STAT1`: raw counts say **×4.5 *down***, CPM says **×1.35 up**. Without normalization we would miss the induction of `IDO1` and conclude the opposite for `STAT1`.

</details>

## Exercise 6 · The logarithm: taming the scale
⏱ *8 min*

CPM fixes the differences between **samples**, but not between **genes**: some genes have <1 CPM and others >10,000, the values are very **skewed**, and the **variance grows with the mean**.
Many analyses (PCA, tests…) are driven by variance, so a handful of abundant genes would decide everything. The **logarithm** compresses large values and turns ratios into differences
(with log₂: doubling = +1, halving = −1). Because log(0) does not exist, we first add a small **pseudocount** (usually 1): **log₂(CPM + 1)**.

**What to do:** the top panels show one gene in one cell type; the bottom panels show **all genes** (variance against mean). Change the **base** of the logarithm and the **pseudocount**.

In [ ]:
#@title ▶ Exercise 6 · The logarithm { display-mode: "form" }
try:
    exercise_logtransform()
except NameError:
    print('⚠️ Run the Setup cell at the top of the notebook first (▶), wait for ✅ Ready, and then run this cell again.')


#### ❓ Questions
1. Compare ① and ② for `CXCL10` in monocytes. What happens to the shape of the distribution and to its variance?
2. Compare ③ and ④: which share of the total variance do the 20 most variable genes hold, before and after the logarithm? What would that mean for a PCA?
3. Change the base to 10 and to *e*. What changes and what does not? And with the pseudocount at 0 and at 10?

<details>
<summary>💡 Answers</summary>

1. In CPM the distribution is **skewed** (long tail: mean 1,297, median 634, skewness 1.6) and its variance is huge (2,350,366). After log₂ it is much more symmetric (skewness −0.3) and the variance is 4.7.
2. In CPM **20 genes hold 41 % of the whole variance**; after log₂(CPM + 1) only **3 %**. Without the logarithm, a PCA would be decided by a few abundant genes (and would mostly "see" them); after it, thousands of genes contribute.
3. The **shape does not change**, only the numbers on the axis (log₂ x = 3.32 × log₁₀ x): the variances are multiplied by 0.48 (ln) or 0.09 (log₁₀). The base is a matter of convention; we use 2 because a difference of 1 means "twice as much". With pseudocount **0**, zeros cannot be transformed and are dropped. With **10**, low values are squashed together. 1 is the usual compromise.

</details>

---
# Part D · Finding patterns in the data
From now on we use the **normalized** values: log₂(CPM + 1).

## Exercise 7 · Heatmaps: genes × samples
⏱ *10 min*

A **heatmap** shows many genes (rows) in many samples (columns) at once. To compare patterns, every gene (row) is **scaled across the samples** (z-score): **red** = above the average of *that* gene, **blue** = below.
The two bars on top show the **cell type** and the **library size** of each column (dark = deep).

**What to do:** the default shows cell-type markers and cytokine-response genes in the four cell types with PBS and four cytokines (12 donors each), with the **raw counts** on the left and the **normalized** values on the right.
Switch between the views, choose other **gene groups**, add your own genes (e.g. `TNF`, `IL1B`, `IL13RA1`) and select other **cytokines** (Ctrl/⌘ + click).

In [ ]:
#@title ▶ Exercise 7 · Heatmaps { display-mode: "form" }
try:
    exercise_heatmaps()
except NameError:
    print('⚠️ Run the Setup cell at the top of the notebook first (▶), wait for ✅ Ready, and then run this cell again.')


#### ❓ Questions
1. Compare BEFORE and AFTER. Which columns are the brightest in the raw heatmap, for almost every gene? Can you read the cell types in each panel?
2. In the normalized heatmap, which cytokine switches on which block of genes, and in which cell type?
3. Select **IFN-beta** and the group *Type I interferon response*. In which cell types do these genes respond? Then add `IL13RA1` and `IL4R` as extra genes: which cell type expresses `IL13RA1`? (keep this in mind for Exercise 12)

<details>
<summary>💡 Answers</summary>

1. **Before**, the brightest columns are the **PBS** ones (dark in the library-size bar), whatever the gene: the picture shows *how many counts each sample has*, and the cell types are hard to read. **After**, the PBS glow is gone and every cell type shows its own markers: `LYZ`, `CD14`, `S100A9`, `FCN1` in monocytes; `CD3E`, `CD3D` in T cells; `CD8A`, `CD8B` in CD8 T cells; `NKG7`, `GNLY`, `KLRF1` in NK cells.
2. **IL-4** → `ALOX15`, `CCL17`, `FCER2` almost only in **monocytes**. **IFN-γ** → `GBP1`, `IDO1`, `CXCL9`, `CXCL10`, strongest in **monocytes**. **IL-1β** → `IL6`, `CXCL8`, `CCL2` in **monocytes**. **IL-2** → `IL2RA`, `GZMB`, `CISH` in **NK and T cells**, the lymphocytes that carry the IL-2 receptor.
3. The interferon-stimulated genes (`ISG15`, `IFIT1`, `MX1`…) go up **in all four cell types** with IFN-β (log₂ fold change between 2 and 3), because the type I interferon receptor is present in all of them. `IL13RA1` (a subunit of the IL-13 receptor) is expressed in **monocytes** (~256 CPM) and practically absent from T and NK cells (~2 CPM).

</details>

> ⚠️ **z-scores are for visualization.** They hide the absolute level of each gene and exaggerate genes that barely change. Never compute fold changes from them.

## Exercise 8 · PCA: the idea, with just two genes
⏱ *10 min*

**PCA** (principal component analysis) summarizes many genes in a few new axes. To see how, take just **two genes**: each sample is a point in a plane (*x* = gene 1, *y* = gene 2).
1. Draw an **axis** (a line through the centre of the cloud) and **project** every sample onto it (drop a perpendicular from the point to the line).
2. The **spread (variance)** of the projected points tells how much information that axis keeps.
3. **Rotate** the axis until the variance is **maximal**: that axis is **PC1**. The perpendicular one is **PC2**.

PCA is *just a rotation* of the axes (no information is lost); **dimensionality reduction** appears when we keep only the first PCs. With thousands of genes the recipe is exactly the same.

**What to do:** there are two fixed examples in monocytes: a **correlated** pair of genes (`ISG15` and `MX1`, both switched on by interferon) and an **uncorrelated** pair (`IDO1` and `IL1B`, two different pathways).
Rotate the axis with the slider and watch panels ①②③. Tick *show PC1 and PC2*, or press *jump to PC1*.

In [ ]:
#@title ▶ Exercise 8 · PCA, the idea { display-mode: "form" }
try:
    exercise_pca_geometry()
except NameError:
    print('⚠️ Run the Setup cell at the top of the notebook first (▶), wait for ✅ Ready, and then run this cell again.')


#### ❓ Questions
1. **Correlated pair**: at which angle does the axis keep the most variance? How much? What does the axis look like compared with the cloud of points?
2. **Uncorrelated pair**: how much does the kept variance change with the angle? Could one axis summarize both genes?
3. In panel ③, at the angle of PC1, what is the correlation between the two new coordinates? If we drop PC2, how much variance do we lose, in each pair?

<details>
<summary>💡 Answers</summary>

1. At about **36°** the axis keeps **98 %** of the total variance: it runs along the long direction of the elongated cloud (the two genes rise and fall together, r = 0.95). One number per sample (the position along PC1) can replace the two genes.
2. Very little: between **45 % and 55 %** whatever the angle (the best axis, PC1, keeps 55 %). The cloud has no preferred direction, so no single axis summarizes both genes: we would lose almost half of the variance.
3. **About 0**: PC1 and PC2 are uncorrelated by construction. Dropping PC2 loses only **2 %** of the variance for the correlated pair but **45 %** for the uncorrelated pair. PCA compresses well *when genes are correlated*: with thousands of genes that respond together (all the interferon genes, all the monocyte genes…) a few PCs summarize most of the variation.

</details>

## Exercise 9 · PCA of all the samples: what drives the variation?
⏱ *8 min*

Each of the 4,224 samples is described by the **2,000 most variable genes**. PCA gives every sample new coordinates (PC1, PC2…), ordered by the amount of variance they hold.
PCA is **unsupervised**: it knows nothing about donors, cell types or cytokines. To find out what a PC *means*, we **colour the samples by their metadata** and see which colouring lines up with the axes.
The **R²** panel quantifies it: the fraction of the variance of each PC that a variable explains (1 = completely).

**What to do:** the plot starts coloured by **donor**. Change the colouring (cell type, control vs treated, cytokine, number of cells, library size, a gene) and the PCs on the axes.

In [ ]:
#@title ▶ Exercise 9 · PCA of all samples { display-mode: "form" }
try:
    exercise_pca_all()
except NameError:
    print('⚠️ Run the Setup cell at the top of the notebook first (▶), wait for ✅ Ready, and then run this cell again.')


#### ❓ Questions
1. With the default colouring (donor), do donors separate on PC1 and PC2? Colour by **cell type**: what do PC1 and PC2 separate, and how much variance do they hold?
2. Which PC best separates the donors? (try different PCs on the axes: the R² panel tells you how much each one has to do with the donors; then colour by donor.) And which PC has the largest R² with the cytokines?
3. Do cytokines have no effect on the cells, then? What does PC2's R² with the *library size* mean: should we worry?

<details>
<summary>💡 Answers</summary>

1. No: donors are mixed. Coloured by cell type, **PC1 (66 % of the variance)** separates the **monocytes** from the lymphocytes and **PC2 (17 %)** separates the **NK cells** from the T cells; CD4 and CD8 T cells are the closest pair. The R² of cell type with PC1 and PC2 is 0.99 and 0.98.
2. The donors separate on **PC6 (R² = 0.92)**, PC3 (0.72) and PC8 (0.65): try PC3 vs PC6. The cytokines have a modest R² on any single PC (at most 0.13, on PC3 and PC10): they matter, but on small PCs.
3. **No!** PCA shows the **largest** differences first, and the difference between a monocyte and a T cell is much bigger than the effect of any single cytokine. Cytokine effects live in later PCs or **inside each cell type** (next exercise). The R² of PC2 with the library size (0.43) is the cell type in disguise: NK cells simply have smaller libraries. After the normalization of Part C, depth is not driving the PCs.

</details>

## Exercise 10 · PCA within one cell type: cytokine responses
⏱ *10 min*

The cell-type differences hide almost everything else. A common remedy is to analyse **one cell type at a time**: we select its samples, choose its own most variable genes and run PCA again.
Within a cell type the donors and the cytokines are what varies.

**What to do:** choose a **cell type** (the default is monocytes, the most responsive) and the **PCs**. Colour by **cytokine** and tick the ones to highlight (PBS is always black), or colour by donor or by a gene.
The bar plots on the right show the **genes with the largest weights** (loadings) on each axis: they tell you what the axis *means*.

In [ ]:
#@title ▶ Exercise 10 · PCA within a cell type { display-mode: "form" }
try:
    exercise_pca_celltype()
except NameError:
    print('⚠️ Run the Setup cell at the top of the notebook first (▶), wait for ✅ Ready, and then run this cell again.')


#### ❓ Questions
1. In the default plot (PC1 vs PC4 in monocytes), which cytokines move away from the main cloud, and in which direction? Look at the genes behind each axis.
2. Switch the axes to PC1 vs PC2 and colour by **donor**. What dominates the first PCs within a cell type? Compare the R² of donor and cytokine in the table.
3. Colour by the expression of `UTY` (a Y-chromosome gene) and try several PCs: which PC separates male from female donors in monocytes? (the paper reports 6 males and 6 females)

<details>
<summary>💡 Answers</summary>

1. **IFN-β** moves to the right, along PC1, where the top genes include interferon-induced genes (`ACOD1`, `CCL8`, `CCL7`, `IDO1`); **IL-4 and IL-13** go up along PC4, whose top genes include `ALOX15` and `WNT5B` (IL-4 targets). **IL-4 and IL-13 land on the same place**: keep this in mind for Exercise 12.
2. **Donors**: the baseline differences between people are the largest variation within a cell type (R² of donor 0.72 on PC1 and 0.81 on PC2, vs 0.18 and 0.13 for cytokine). The cytokines show up more clearly in later PCs (R² 0.32 on PC3, 0.38 on PC4): the individual effects are smaller than the differences between people. That is why every cytokine will be compared with the PBS of the **same donor**.
3. **PC5** (correlation 0.87 between `UTY` and PC5). The sex of the donor is one of the largest sources of variation in monocytes, even if it has nothing to do with the cytokines. (In the other cell types it appears on another PC: PC2 in CD4 T cells, PC3 in CD8 T and NK cells.)

</details>

---
# Part E · Differential expression: which genes change?

PCA showed that cytokines move the samples in particular directions. **Differential expression (DE)** asks the question **gene by gene**:

> *Is the expression of gene g with the cytokine different from its expression with PBS, by more than we would expect by chance?*

The results you will explore were **computed beforehand** with **DESeq2** (nothing is run now): for each of the 4 cell types and each of the 87 cytokines, the comparison *cytokine vs PBS* in the 12 donors.

> 🧬 **Only the immune genes.** To keep the analysis light and the plots readable, DE was run on **~500 immune-system genes** (506 expressed ones): the union of the GO sets *cytokine-mediated signaling pathway* and *T cell activation* and the Hallmark *interferon α / γ response* sets. Each comparison tests ~450 of them.
> **What do we lose?** (1) The list was built *from genes that respond to cytokines*, so the **fraction of DEGs is much higher** than in a whole-transcriptome analysis; do not compare percentages. (2) **Genes outside the list cannot be found**, even classic ones (`ACTB`, `GBP1`, `ALOX15`, `NKG7`, `CCL17`… are not in it): the tools tell you when a gene was not analysed.

<details>
<summary>🔬 How were the results computed? (optional)</summary>

* DESeq2 works on the **raw counts** and normalizes them itself (a robust version of CPM).
* The **design** `~ donor + cytokine` is **paired**: every donor is compared with itself, so differences between people cancel out.
* For each gene it fits a **negative binomial** model (counts are not normally distributed), and a **Wald test** asks whether the effect of the cytokine is larger than the noise.
* Because ~450 genes are tested at once, the p-values are **adjusted for multiple testing** (Benjamini–Hochberg, controlling the False Discovery Rate): the **adjusted p-value** is the one to use.
* Genes with too few counts (fewer than 10 counts in 12 of the 24 samples) are not tested.

</details>

Two numbers describe every gene:

| Number | Meaning |
|---|---|
| **log₂ fold change** | the **size** of the change: +1 = doubles (×2), +2 = ×4, −1 = halves |
| **adjusted p-value** | the **evidence**: how surprising the change would be if the cytokine did nothing (small = convincing) |

A gene is called **differentially expressed (DEG)** when it passes **both** thresholds.


## Exercise 11 · Volcano plots: the genes that respond
⏱ *12 min*

A **volcano plot** shows every gene at once: *x* = log₂ fold change (effect size), *y* = −log₁₀ adjusted p-value (evidence; higher = more convincing).
**Red** = up-regulated DEGs, **blue** = down-regulated DEGs, **grey** = not significant. Genes at the top and far from the centre are the most robust and strongest.

**What to do:** the default is **IFN-γ in monocytes**, with the genes `CXCL9`, `CXCL10`, `IDO1`, `STAT1` and `IRF1` highlighted. Change the **cell type** and the **cytokine**, the **thresholds** and the genes to highlight
(type a symbol, or pick a ready-made list; only the ~500 immune genes can be found); the tables show their fold change, adjusted p-value and mean expression.

In [ ]:
#@title ▶ Exercise 11 · Volcano plots { display-mode: "form" }
try:
    exercise_volcano()
except NameError:
    print('⚠️ Run the Setup cell at the top of the notebook first (▶), wait for ✅ Ready, and then run this cell again.')


#### ❓ Questions
1. How many genes go up and down with IFN-γ in monocytes? Look at the five highlighted genes: which are DEGs, and how many times higher than in PBS? Is any of them significant but *not* a DEG?
2. Set the fold-change threshold to 0, 1 and 2 (adjusted p-value 0.05), and then change the adjusted p-value threshold from 0.2 to 0.001 (fold change 0.5). Which threshold matters more? What does the fold-change threshold buy us?
3. Change the cytokine to **IL-4** and highlight `CCL26`, `CD1C`, `STAT1`, `IDO1`. What happens to the interferon genes?
4. Choose **NK cells** and **IL-2**. What do the top up-regulated genes `IL2RA`, `CISH` and `SOCS2` have in common?

<details>
<summary>💡 Answers</summary>

1. **96 up and 95 down** (of 452 genes tested). `CXCL9` (×10.5), `IDO1` (×5.5) and `CXCL10` (×3.7) are strongly up: the classic **IFN-γ response** (T-cell recruitment and tryptophan catabolism). `STAT1` is **not** a DEG (×1.1, adjusted p = 0.34), although it is the transcription factor of that response: it is already abundant in monocytes. `IRF1` is **significant** (adjusted p ≈ 10⁻⁵) but its change is small (×1.4, log₂FC 0.49 < 0.5): significant is not the same as large.
2. Fold-change threshold 0 / 1 / 2 → **311 / 70 / 14** DEGs; adjusted p-value 0.2 / 0.001 → 197 / 161 DEGs (with fold change 0.5; 191 with 0.05). The **fold-change threshold matters much more**: it removes statistically significant but biologically small changes, keeping those that are both reliable and large.
3. IL-4 strongly up-regulates its own targets (`CCL26` ×207, `CD1C` ×21) and **down-regulates** the interferon genes (`IDO1` ×0.1, `STAT1` ×0.4). IL-4 and IFN-γ push monocytes in opposite directions.
4. They are **IL-2 signalling genes**: `IL2RA` is the high-affinity IL-2 receptor α chain and `CISH` and `SOCS2` are STAT5 targets that feed back to switch the signal off. NK cells respond to IL-2 through STAT5.

</details>


## Exercise 12 · Comparing two responses: similar or different?
⏱ *10 min*

Do two cytokines switch on the same genes? For every gene we have a log₂ fold change in **comparison A** and in **comparison B** (each = a cytokine vs PBS in one cell type). In a scatter plot of A against B,
two similar responses fall along the **diagonal**, and the **correlation coefficient** summarizes it: **r = 1** same response, **r = 0** unrelated, **r = −1** opposite.
Colours show where the gene is significant: **blue** only in A, **orange** only in B, **violet** in both, grey in neither. The right panel ranks all the other cytokines by how much they resemble A (in the same cell type).

**What to do:** the default compares **IL-4 and IL-13 in monocytes**. Change the cytokine and the cell type of A and B (they can be in different cell types!), load one of the **examples**, choose which genes are used for the correlation and **highlight genes**.

> ⚠️ Both comparisons use the **same PBS control**: noise in the PBS samples ends up in both and can create or inflate a correlation. Be careful with small values of r (≈ 0.2 or less).

In [ ]:
#@title ▶ Exercise 12 · Comparing two responses { display-mode: "form" }
try:
    exercise_compare()
except NameError:
    print('⚠️ Run the Setup cell at the top of the notebook first (▶), wait for ✅ Ready, and then run this cell again.')


#### ❓ Questions
1. How similar are the responses to IL-4 and IL-13 in monocytes? Now change both comparisons to **CD4 T cells**. What happens, and why? (remember `IL13RA1` from Exercise 7)
2. Load the examples **IFN-β vs IFN-γ in monocytes** and **IFN-γ vs IL-4 in monocytes**. Compare the correlations and use the right-hand panel: which cytokines respond most like IFN-γ?
3. Load **TNF-α in monocytes vs in CD4 T cells** and **IL-6 in monocytes vs in CD4 T cells**: which stimulus triggers a more similar response in the two cell types? And **TNF-α vs IL-6** in monocytes?
4. Change *genes used for the correlation* to "genes significant in A or B". When does r change a lot, and when not?

<details>
<summary>💡 Answers</summary>

1. In monocytes the two responses are **almost identical (r = 0.97)**, and IL-13 is the cytokine most similar to IL-4 out of 86. In CD4 T cells **r falls to 0.41** (IL-13 is only 3rd). IL-4 and IL-13 share a receptor (IL-4Rα/IL-13Rα1), but T cells hardly express `IL13RA1` (~2 CPM vs ~256 in monocytes), so they respond to IL-4 and not to IL-13. The **same pair of stimuli can look identical or very different depending on the cell type**.
2. **IFN-β vs IFN-γ: r = 0.78** (both interferons activate STAT1 and the interferon-stimulated genes); **IFN-γ vs IL-4: r = −0.35**: opposite programmes (what one raises, the other lowers). The ranking for IFN-γ in monocytes starts with IL-12 (0.87), IFN-β and IFN-ω (0.78) and IL-15 (0.74), and ends with IL-17E and TNF-α (−0.46, −0.45).
3. **TNF-α** gives a similar response in the two cell types (**r = 0.70**), **IL-6** a more cell-type-specific one (**r = 0.47**). **TNF-α vs IL-6 in monocytes: r = 0.22**, a weak relation. Always look at the data before assuming that two cytokines share a program.
4. In monocytes, where both cytokines change many genes, r hardly moves (IL-4 vs IL-13: 0.97 → 0.97). Where responses are weaker, restricting to genes that respond (CD4 T cells: 0.41 → 0.51; TNF-α vs IL-6 in monocytes: 0.22 → 0.28) removes noise and r **increases**. But it also *selects* genes that change, so the comparison is no longer free of bias: report which genes were used. Remember too that the whole analysis already uses a list of immune genes, enriched in cytokine-responsive genes: correlations are higher than they would be genome-wide.

</details>


---
# 🏁 Wrap-up

You went from a single variable to the comparison of transcriptional programs:

| Exercise | The idea to keep |
|:--:|---|
| 1–2 | A variable is a name for a value; a plot type answers a question. |
| 3 | AnnData keeps the numbers (`X`), the samples (`obs`) and the genes (`var`) **aligned**; `adata[samples, genes]` selects a subset. |
| 4 | Raw counts mix biology with the **amount of material** (library size). |
| 5 | **CPM** divides by the library size so that samples become comparable. |
| 6 | The **logarithm** compresses the scale, turns ratios into differences and stops a few genes from dominating. |
| 7 | **Heatmaps** show patterns; they only make sense *after* normalizing. |
| 8–10 | **PCA** is a rotation that finds the directions of largest variance. Colouring by metadata tells what each PC means: cell type ≫ donor ≫ cytokine. Look **inside a cell type** to see cytokine effects. |
| 11 | A **DEG** needs evidence (**adjusted p-value**) *and* a relevant effect (**fold change**). The paired design compares each donor with itself. |
| 12 | The **correlation of fold changes** says whether two stimuli trigger similar programs, and it **depends on the cell type**. |

### Where to go from here
* **Functional enrichment** of the lists of DEGs (which pathways?): `gseapy`, `decoupler`.
* **Single-cell analysis** of the original data: `scanpy`; best practices in Heumos *et al.* (2023).
* The **Human Cytokine Dictionary** itself and its `huCIRA` package (Oesinghaus *et al.*, 2025) to interpret your own data with cytokine-response signatures.

### 📝 Quick quiz
Your instructor will show the **QR code** of the quiz (3 questions, 1 minute).

### References
* Oesinghaus L. *et al.* (2025) A single-cell cytokine dictionary of human peripheral blood. *bioRxiv*, doi:10.64898/2025.12.12.693897
* Virshup I. *et al.* (2024) anndata: Access and store annotated data matrices. *J. Open Source Softw.* 9:4371
* Heumos L. *et al.* (2023) Best practices for single-cell analysis across modalities. *Nat. Rev. Genet.* 24:550–572
* Love M.I., Huber W., Anders S. (2014) Moderated estimation of fold change and dispersion for RNA-seq data with DESeq2. *Genome Biol.* 15:550
* Muzellec B. *et al.* (2023) PyDESeq2: a python package for bulk RNA-seq differential expression analysis. *Bioinformatics* 39:btad547
* Benjamini Y., Hochberg Y. (1995) Controlling the false discovery rate. *J. R. Stat. Soc. B* 57:289–300
